# HERMES TwinPast historical-evidence diagnostic

Upload **this notebook only** to your AutoDL JupyterLab and choose **Restart Kernel → Run All**.
The same RTX 4090 container is fastest because it can reuse the validated causal-pilot Python
environment and model cache. A fresh CUDA 12.8-compatible container can bootstrap the same pins.
The notebook embeds the pinned model wrapper and experiment files. It does not need a Git clone.

The run tests nine frozen development pairs with two option orders. Each pair has four conditions
per order: question-only, recent-four, two earlier A plus recent-four, and two earlier B plus
recent-four. The history arms share exactly the same recent features, question and answer prompt
within an option order. Their scores are mapped to canonical answer identities.

Only 18 short MP4s are needed. Individual clips use bounded requests and verified SHA-256 hashes,
and completed pairs are checkpointed. If the exact clips already live on AutoDL, set MEDIA_REUSE
to a directory containing <pair_id>/A.mp4 and <pair_id>/B.mp4. Otherwise leave the defaults.
The first pair runs GPU preflight checks before the diagnostic proceeds.

Watch for MEDIA_READY, TWINPAST_PREFLIGHT_PASS and PAIR_DONE. At TWINPAST_DIAGNOSTIC_COMPLETE,
download the evidence ZIP and save the executed notebook. This is a diagnostic with annotated
earlier frames. No automatic retrieval method is evaluated.

Verified media total: 361.6 MiB.


In [1]:
!source /etc/network_turbo

设置成功!
注意：
1. 仅限学术用途和加速访问github/huggingface，不承诺稳定性
2. 开启加速后对访问其他资源如pip源等会*更慢*


In [2]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, time, zipfile, ast
from datetime import datetime, timezone

WORK = Path('/root/autodl-tmp/hermes_twinpast_diagnostic')
REUSE = Path('/root/autodl-tmp/hermes_causal_pilot')
BASE_REVISION = '2a1385e0abc94debb64761b0465713c799a81c1c'
MODEL_ID = 'llava-hf/llava-onevision-qwen2-0.5b-ov-hf'
MODEL_SHA = '74dd0bf867a4cda7950c17663794267c60cf4b40'
TF_SHA = '66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
VALIDATE_ONLY = os.environ.get('HERMES_TWINPAST_VALIDATE_ONLY') == '1'
assert VALIDATE_ONLY or (WORK.is_absolute() and REUSE.is_absolute())
assert WORK != REUSE and not WORK.is_relative_to(REUSE) and not REUSE.is_relative_to(WORK)
REPO = WORK/'pinned_source'
VENV = WORK/'venv'
PYTHON = VENV/'bin/python'
RUNNER = WORK/'provenance/twinpast_runner.py'
print('Plan: 9 pairs, 72 forced-option score rows')
print('WORK:', WORK, '\nReuse candidate:', REUSE)
print('Mode:', 'CPU notebook validation (no network/GPU)' if VALIDATE_ONLY else 'AutoDL execution')
MEDIA_REUSE = Path('/root/autodl-tmp/twinpast-development')


Plan: 9 pairs, 72 forced-option score rows
WORK: /root/autodl-tmp/hermes_twinpast_diagnostic 
Reuse candidate: /root/autodl-tmp/hermes_causal_pilot
Mode: AutoDL execution


## 1. Verify runtime and materialize the experiment


In [3]:
def network_environment():
    env = os.environ.copy()
    if Path('/etc/network_turbo').is_file():
        proc = subprocess.run(['bash', '-lc', 'source /etc/network_turbo >/dev/null 2>&1; env -0'], capture_output=True)
        if proc.returncode == 0:
            for item in proc.stdout.split(b'\0'):
                if b'=' in item:
                    key, value = item.split(b'=', 1)
                    env[key.decode()] = value.decode()
    env.update(HF_HOME=str((REUSE if (REUSE/'cache/huggingface').is_dir() else WORK)/'cache/huggingface'), HF_HUB_DISABLE_XET='1',
               HF_HUB_ENABLE_HF_TRANSFER='0', TOKENIZERS_PARALLELISM='false',
               CUDA_VISIBLE_DEVICES='0', OMP_NUM_THREADS='8', GIT_LFS_SKIP_SMUDGE='1',
               UV_CACHE_DIR=str(WORK/'cache/uv'), UV_LINK_MODE='copy', TMPDIR=str(WORK),
               UV_DEFAULT_INDEX='https://pypi.tuna.tsinghua.edu.cn/simple',
               PYTORCH_ALLOC_CONF='expandable_segments:True')
    env.pop('PYTHONHOME', None)
    env['PYTHONPATH'] = ''
    return env

def logged(command, name, env=None, cwd=None):
    import queue, threading
    logfile = WORK/'logs'/name
    logfile.parent.mkdir(parents=True, exist_ok=True)
    print('SETUP_STAGE', name, flush=True)
    with logfile.open('a', encoding='utf-8') as handle:
        proc = subprocess.Popen([str(x) for x in command], env=env, cwd=cwd,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
        lines = queue.Queue()
        def collect():
            try:
                for line in proc.stdout:
                    lines.put(line)
            finally:
                lines.put(None)
        reader = threading.Thread(target=collect, daemon=True)
        reader.start()
        started = time.monotonic()
        while True:
            try:
                line = lines.get(timeout=30)
            except queue.Empty:
                line = f'COMMAND_ALIVE log={name} elapsed_min={(time.monotonic()-started)/60:.1f} (liveness only)\n'
            if line is None:
                break
            print(line, end='', flush=True)
            handle.write(line)
            handle.flush()
        reader.join(timeout=5)
        status = proc.wait()
        proc.stdout.close()
    if status:
        raise RuntimeError(f'Command failed ({status}); see {logfile}. Failure evidence, if created, stays in WORK.')

def probe(command, env=None, cwd=None):
    return subprocess.run([str(x) for x in command], env=env, cwd=cwd,
                          text=True, capture_output=True)

def package_environment(base, index='https://pypi.tuna.tsinghua.edu.cn/simple'):
    # AutoDL turbo is for GitHub/Hugging Face. Domestic package mirrors should
    # bypass those proxies, especially for large NVIDIA dependency wheels.
    direct = dict(base)
    for key in ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
                'http_proxy', 'https_proxy', 'all_proxy', 'no_proxy'):
        direct.pop(key, None)
    direct.update(UV_HTTP_TIMEOUT='600', UV_HTTP_RETRIES='5',
                  UV_CONCURRENT_DOWNLOADS='2', UV_DEFAULT_INDEX=index,
                  PIP_DEFAULT_TIMEOUT='600', PIP_RETRIES='5')
    return direct

def install_packages(uv, python, packages, env, extra=()):
    errors = []
    for index in ('https://pypi.tuna.tsinghua.edu.cn/simple', 'https://mirrors.aliyun.com/pypi/simple'):
        try:
            logged(uv + ['pip', 'install', '--python', python, '--index-url', index]
                   + list(extra) + list(packages), 'setup.log', package_environment(env, index))
            return
        except RuntimeError as exc:
            errors.append(type(exc).__name__)
            print('Package download failed; trying alternate domestic index with cached files', flush=True)
    raise RuntimeError('Package installation failed on both indexes; cached files are preserved. See setup.log.')

if not VALIDATE_ONLY:
    assert sys.platform.startswith('linux'), 'Run this notebook in an AutoDL Linux container'
    WORK.mkdir(parents=True, exist_ok=True)
    ENV = network_environment()
    free = shutil.disk_usage(WORK).free / 1024**3
    minimum_free = 1 if (REUSE/'venv/bin/python').is_file() and (REUSE/'cache/huggingface').is_dir() else 18
    assert free >= minimum_free, f'Need {minimum_free} GiB free on the data disk; found {free:.1f} GiB'
    smi = probe(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], ENV)
    assert smi.returncode == 0 and smi.stdout.strip(), 'CUDA GPU is missing'
    print('GPU:', smi.stdout.strip(), '\nFree GiB:', round(free, 1))
    import re
    first_gpu = smi.stdout.splitlines()[0]
    assert float(re.search(r'(\d+(?:\.\d+)?)\s*MiB', first_gpu)[1]) >= 15000, 'Use a GPU with at least 15 GiB VRAM'
    # Never print the environment or proxy values.


GPU: NVIDIA GeForce RTX 4090, 24564 MiB, 580.105.08 
Free GiB: 14.2


In [4]:
SOURCES = {'pinned_source/inference/llavaov_hermes.py': 'import re\nimport time\nimport torch\nimport torch.nn.functional as F\nimport types\nfrom transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor, DynamicCache\nfrom logzero import logger\n\nfrom inference.abstract_hermes import Abstract_Hermes\nfrom inference.reindex_1d import (\n    _get_rotary_module,\n    get_cache_seq_len,\n    contiguous_kv,\n    compute_cos_sin_for_positions,\n    rotary_delta,\n    apply_rotary_delta_to_keys_only,\n    apply_rotary_pos_emb_1d,\n)\n\n\nclass LlavaOneVision_Hermes(LlavaOnevisionForConditionalGeneration, Abstract_Hermes):\n    def __init__(self, config, processor, init_prompt_ids, kv_size, streaming=True):\n        super().__init__(processor, init_prompt_ids, kv_size)\n        self.streaming = streaming\n        \n        num_layers = config.num_hidden_layers if hasattr(config, \'num_hidden_layers\') else 32\n        self.num_layers = num_layers\n        \n        self.short_term_ratio = 0.3\n        self.long_term_ratio = 0.3\n        self.short_term_threshold = int(self.num_layers * self.short_term_ratio)\n        self.long_term_threshold = int(self.num_layers * (1 - self.long_term_ratio))\n        \n        self._position_ids_cache = [None for _ in range(num_layers)]\n        self.token_activity_cache = [None for _ in range(num_layers)]\n        \n        self._layer_position_ids = {}\n        self._hook_handles = []\n        self._register_forward_hooks()\n        \n        self.total_processed_frames = 0\n        \n        self._patch_rotary_embeddings()\n\n    def _patch_rotary_embeddings(self):\n        """Patch rotary embedding to support dynamic resizing based on logical position IDs"""\n        def patched_forward(rotary_self, x, seq_len=None):\n            required_len = getattr(rotary_self, \'_required_seq_len\', 0)\n            if seq_len is not None and required_len > seq_len:\n                seq_len = required_len\n            return rotary_self._original_forward(x, seq_len=seq_len)\n\n        if hasattr(self.language_model.model, \'layers\'):\n            for layer in self.language_model.model.layers:\n                if hasattr(layer, \'self_attn\') and hasattr(layer.self_attn, \'rotary_emb\'):\n                    rotary_emb = layer.self_attn.rotary_emb\n                    if not hasattr(rotary_emb, \'_original_forward\'):\n                        rotary_emb._original_forward = rotary_emb.forward\n                        rotary_emb.forward = types.MethodType(patched_forward, rotary_emb)\n                        rotary_emb._required_seq_len = 0\n\n    def _set_rotary_required_len(self, required_len):\n        """Set required sequence length for rotary embeddings"""\n        if hasattr(self.language_model.model, \'layers\'):\n            for layer in self.language_model.model.layers:\n                if hasattr(layer, \'self_attn\') and hasattr(layer.self_attn, \'rotary_emb\'):\n                    layer.self_attn.rotary_emb._required_seq_len = required_len\n\n    def _register_forward_hooks(self):\n        if not hasattr(self.language_model, \'model\') or not hasattr(self.language_model.model, \'layers\'):\n            return\n        \n        def create_hook(layer_idx):\n            def hook(module, args, kwargs):\n                if layer_idx in self._layer_position_ids:\n                    kwargs[\'position_ids\'] = self._layer_position_ids[layer_idx]\n                return args, kwargs\n            return hook\n        \n        for layer_idx, layer in enumerate(self.language_model.model.layers):\n            handle = layer.register_forward_pre_hook(create_hook(layer_idx), with_kwargs=True)\n            self._hook_handles.append(handle)\n    \n    def _clear_forward_hooks(self):\n        for handle in self._hook_handles:\n            handle.remove()\n        self._hook_handles = []\n\n    def _prune_activity_cache(self, keep_indices_per_layer):\n        if all(c is None for c in self.token_activity_cache):\n            return\n\n        for layer_idx in range(self.num_layers):\n            if self.token_activity_cache[layer_idx] is not None:\n                keep_indices = keep_indices_per_layer[layer_idx]\n                if not isinstance(keep_indices, torch.Tensor):\n                    keep_indices = torch.tensor(keep_indices, device=self.device, dtype=torch.long)\n                \n                curr_len = self.token_activity_cache[layer_idx].shape[0]\n                keep_indices = keep_indices[(keep_indices >= 0) & (keep_indices < curr_len)]\n                \n                if keep_indices.numel() > 0:\n                    self.token_activity_cache[layer_idx] = self.token_activity_cache[layer_idx][keep_indices]\n                else:\n                    self.token_activity_cache[layer_idx] = torch.zeros(0, device=self.device)\n\n    def _append_position_ids_layer(self, layer_idx: int, start: int, length: int):\n        device = self.device\n        new_pos = torch.arange(start, start + length, device=device, dtype=torch.long)\n        if self._position_ids_cache[layer_idx] is None:\n            self._position_ids_cache[layer_idx] = new_pos\n        else:\n            self._position_ids_cache[layer_idx] = torch.cat(\n                [self._position_ids_cache[layer_idx], new_pos], dim=0\n            )\n    \n    def _append_position_ids(self, start_per_layer, length: int):\n        if isinstance(start_per_layer, int):\n            for layer_idx in range(self.num_layers):\n                self._append_position_ids_layer(layer_idx, start_per_layer, length)\n        else:\n            for layer_idx in range(self.num_layers):\n                self._append_position_ids_layer(layer_idx, start_per_layer[layer_idx], length)\n\n    def _get_cache_seq_len_per_layer(self) -> list:\n        if self.kv_cache is None:\n            return [0] * self.num_layers\n        \n        lengths = []\n        for layer_idx in range(len(self.kv_cache)):\n            k_layer, v_layer = self.kv_cache[layer_idx]\n            lengths.append(k_layer.shape[2])\n        return lengths\n\n    def _get_next_start_pos_per_layer(self) -> list:\n        next_pos = []\n        for layer_idx in range(self.num_layers):\n            if (layer_idx < len(self._position_ids_cache) and \n                self._position_ids_cache[layer_idx] is not None and \n                self._position_ids_cache[layer_idx].numel() > 0):\n                next_pos.append(self._position_ids_cache[layer_idx][-1].item() + 1)\n            else:\n                next_pos.append(0)\n        return next_pos\n    \n    def _sanitize_keep_indices(self, keep_indices_1d: torch.Tensor, seq_len: int) -> torch.Tensor:\n        keep_indices_1d = keep_indices_1d.to(self.device, dtype=torch.long)\n        keep_indices_1d = keep_indices_1d[(keep_indices_1d >= 0) & (keep_indices_1d < seq_len)]\n        if keep_indices_1d.numel() == 0:\n            return torch.tensor([0], device=self.device, dtype=torch.long)\n        keep_indices_1d = torch.unique(keep_indices_1d, sorted=True)\n        return keep_indices_1d\n\n    def _build_position_ids(self, past_len: int, q_len: int, batch: int) -> torch.Tensor:\n        pos_1d = torch.arange(past_len, past_len + q_len, device=self.device, dtype=torch.long)\n        position_ids = pos_1d.view(1, -1).expand(batch, -1)\n        return position_ids\n\n    @torch.inference_mode()\n    def _shrink_positions_and_rerotate_keys(self, keep_indices_per_layer):\n        device = self.device\n        \n        curr_lens = self._get_cache_seq_len_per_layer()\n        \n        for layer_idx in range(self.num_layers):\n            if (self._position_ids_cache[layer_idx] is None or \n                self._position_ids_cache[layer_idx].numel() != curr_lens[layer_idx]):\n                self._position_ids_cache[layer_idx] = torch.arange(\n                    curr_lens[layer_idx], device=device, dtype=torch.long\n                )\n        max_pos_limit = getattr(self.language_model.config, "max_position_embeddings", 32768)\n        compact_threshold = max_pos_limit - 1024\n        \n        current_max_pos = 0\n        for cache in self._position_ids_cache:\n             if cache is not None and cache.numel() > 0:\n                 current_max_pos = max(current_max_pos, cache[-1].item())\n        \n        should_compact = (current_max_pos > compact_threshold) if self.streaming else True\n        \n        if should_compact:\n            logger.info(f"[Shrink] Max position {current_max_pos} > {compact_threshold}. Compacting position IDs (Shift Left).")\n        \n        old_position_ids_cache = [cache.clone() for cache in self._position_ids_cache]\n\n        sample_k = self.kv_cache[0][0]\n        dtype = sample_k.dtype\n        \n        new_kv_cache = []\n        \n        for layer_idx, (k_layer, v_layer) in enumerate(self.kv_cache):\n            \n            keep_indices_layer = keep_indices_per_layer[layer_idx]\n            if not isinstance(keep_indices_layer, torch.Tensor):\n                keep_indices_layer = torch.as_tensor(keep_indices_layer, dtype=torch.long, device=device)\n            \n            seq_len_layer = k_layer.shape[2]\n            safe_idx = self._sanitize_keep_indices(keep_indices_layer, seq_len_layer)\n            \n            if safe_idx.numel() == 0:\n                logger.warning(f"Layer {layer_idx}: After sanitization, keep_indices is empty; keeping first token")\n                safe_idx = torch.tensor([0], device=device, dtype=torch.long)\n\n            is_long_term = (layer_idx >= self.long_term_threshold)\n            \n            # === Extract Kept Tokens ===\n            k_kept = torch.index_select(k_layer, dim=2, index=safe_idx)\n            v_kept = torch.index_select(v_layer, dim=2, index=safe_idx)\n            old_pos_kept = old_position_ids_cache[layer_idx].index_select(0, safe_idx)\n            \n            # === Determine New Positions for Kept Tokens ===\n            if should_compact:\n                new_pos_kept = torch.arange(len(safe_idx), device=device, dtype=torch.long)\n            else:\n                new_pos_kept = old_pos_kept\n\n            # === Handle RoPE for Kept Tokens ===\n            if should_compact:\n                cos_old, sin_old = compute_cos_sin_for_positions(\n                    self.language_model, len(old_pos_kept), old_pos_kept, dtype, device\n                )\n                cos_new, sin_new = compute_cos_sin_for_positions(\n                    self.language_model, len(new_pos_kept), new_pos_kept, dtype, device\n                )\n                cos_delta, sin_delta = rotary_delta(\n                    cos_old, sin_old, cos_new, sin_new\n                )\n                k_kept_final = apply_rotary_delta_to_keys_only(k_kept, cos_delta, sin_delta)\n            else:\n                k_kept_final = k_kept\n            \n            # === Handle Folding (Summary Token) for Long-term Layers ===\n            if is_long_term:\n                mask = torch.ones(seq_len_layer, dtype=torch.bool, device=device)\n                mask[safe_idx] = False\n                prune_indices = torch.nonzero(mask).squeeze(1)\n                \n                if prune_indices.numel() > 0:\n                    k_pruned = torch.index_select(k_layer, dim=2, index=prune_indices)\n                    v_pruned = torch.index_select(v_layer, dim=2, index=prune_indices)\n                    \n                    v_summary = v_pruned.mean(dim=2, keepdim=True)\n                    \n                    old_pos_pruned = old_position_ids_cache[layer_idx].index_select(0, prune_indices)\n                    \n                    if should_compact:\n                        summary_pos_id = len(safe_idx)\n                    else:\n                        summary_pos_id = new_pos_kept[-1].item() + 1\n                    \n                    summary_pos_tensor = torch.tensor([summary_pos_id], device=device, dtype=torch.long)\n                    target_pos_pruned = summary_pos_tensor.repeat(old_pos_pruned.shape[0])\n                    \n                    cos_old, sin_old = compute_cos_sin_for_positions(\n                        self.language_model, len(old_pos_pruned), old_pos_pruned, dtype, device\n                    )\n                    cos_new, sin_new = compute_cos_sin_for_positions(\n                        self.language_model, len(target_pos_pruned), target_pos_pruned, dtype, device\n                    )\n                    \n                    cos_delta, sin_delta = rotary_delta(cos_old, sin_old, cos_new, sin_new)\n                    \n                    k_pruned_aligned = apply_rotary_delta_to_keys_only(k_pruned, cos_delta, sin_delta)\n                    \n                    k_summary_final = k_pruned_aligned.mean(dim=2, keepdim=True)\n                    \n                    k_final = torch.cat([k_kept_final, k_summary_final], dim=2)\n                    v_final = torch.cat([v_kept, v_summary], dim=2)\n                    \n                    new_pos_layer = torch.cat([new_pos_kept, summary_pos_tensor])\n                    \n                else:\n                    k_final = k_kept_final\n                    v_final = v_kept\n                    new_pos_layer = new_pos_kept\n            else:\n                k_final = k_kept_final\n                v_final = v_kept\n                new_pos_layer = new_pos_kept\n\n            new_kv_cache.append((k_final.contiguous(), v_final.contiguous()))\n            \n            self._position_ids_cache[layer_idx] = new_pos_layer.clone()\n        \n        self.kv_cache = new_kv_cache\n        \n        new_lens = self._get_cache_seq_len_per_layer()\n        logger.info(f"Layer-wise shrink completed. Lengths: min={min(new_lens)}, max={max(new_lens)}, "\n                   f"first={new_lens[0]}, last={new_lens[-1]}")\n    \n    @torch.inference_mode()\n    def predict_next_question(self):\n        """\n        Generates queries to drive attention calculation.\n        Includes Semantic Anchors for better coverage.\n        """\n        if hasattr(self, \'conv_history\') and len(self.conv_history) > 0:\n            last_q, last_a, last_options = self.conv_history[-1]\n            \n            option_match = re.match(r\'^\\s*(?:\\()?([A-Z])(?:\\))?\\.?\\s*$\', last_a)\n            if option_match and last_options is not None:\n                option_char = option_match.group(1)\n                option_idx = ord(option_char) - ord(\'A\')\n                if 0 <= option_idx < len(last_options):\n                    last_a = last_options[option_idx]\n\n            last_a = re.sub(r\'[A-Z]\\) \', \'\', last_a)\n\n            last_round_history = f"Question: {last_q} Answer: {last_a}"\n            \n            global_query = f"Context summary: {last_round_history}. Summarize the video narrative, identifying main characters, key events, timeline changes, and the overall theme."\n            local_query = f"Find recent details related to: {last_round_history}. Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships."\n            \n        else:\n            global_query = (\n                "Summarize the video narrative, identifying main characters, key events, timeline changes, and the overall theme."\n            )\n            local_query = (\n                "Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships."\n            )\n            \n        print(f"Local question: {local_query}")\n        print(f"Global question: {global_query}")\n        return local_query, global_query\n    \n    @torch.inference_mode()\n    def encode_init_prompt(self):\n        super().encode_init_prompt()\n        self.total_processed_frames = 0\n        curr_lens = self._get_cache_seq_len_per_layer()\n        for layer_idx in range(self.num_layers):\n            self._position_ids_cache[layer_idx] = torch.arange(\n                curr_lens[layer_idx], device=self.device, dtype=torch.long\n            )\n    \n    def get_video_features(self, pixel_values_videos):\n            batch_size, frames, channels, height, width = pixel_values_videos.shape\n            pixel_values_videos = pixel_values_videos.view(batch_size * frames, channels, height, width)\n            video_features = self.vision_tower(pixel_values_videos, output_hidden_states=True)\n            selected_video_feature = video_features.hidden_states[self.config.vision_feature_layer]\n            if self.config.vision_feature_select_strategy == "default":\n                selected_video_feature = selected_video_feature[:, 1:]\n            elif self.config.vision_feature_select_strategy == "full":\n                selected_video_feature = selected_video_feature\n            video_features = self.multi_modal_projector(selected_video_feature)\n            video_features = self.apply_pooling(video_features)\n            frames_after_merge = video_features.shape[0]\n            video_features = video_features.reshape(batch_size, frames_after_merge * video_features.shape[1], -1)  # (B, Nv*196, D)\n            return video_features\n    \n    @torch.inference_mode()\n    def encode_video_chunk(self, video_chunk):\n        if video_chunk is None or (hasattr(video_chunk, "shape") and video_chunk.shape[0] == 0):\n            return\n\n        if len(video_chunk.shape) == 4 and video_chunk.shape[-1] == 3:\n            video_chunk = video_chunk.permute(0, 3, 1, 2)\n\n        pixel_values_videos = self.processor.video_processor(\n            video_chunk, return_tensors="pt"\n        ).pixel_values_videos.to(self.device, self.dtype)\n        video_features = self.get_video_features(pixel_values_videos)\n\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len = video_features.shape[1]\n        batch = video_features.shape[0]\n        \n        max_start_pos = max(start_pos_per_layer) if start_pos_per_layer else 0\n        self._set_rotary_required_len(max_start_pos + q_len)\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        default_position_ids = self._build_position_ids(start_pos_per_layer[0], q_len, batch)\n        \n        out = self.language_model(\n            inputs_embeds=video_features,\n            past_key_values=self.kv_cache,\n            use_cache=True,\n            return_dict=True,\n            position_ids=default_position_ids,\n            cache_position=None,\n            attention_mask=None,\n        )\n        self.kv_cache = out.past_key_values\n        self.kv_cache = contiguous_kv(self.kv_cache)\n\n        self._append_position_ids(start_pos_per_layer, q_len)\n        self.last_encoded_frames = video_chunk.shape[0]\n        self.total_processed_frames += video_chunk.shape[0]\n        \n        self._layer_position_ids.clear()\n\n    @torch.inference_mode()\n    def apply_kv_cache_pruning_strict(self, keep_indices_all_layers):\n        if self.kv_cache is None:\n            logger.warning("No KV-Cache to prune")\n            return\n        if not keep_indices_all_layers or len(keep_indices_all_layers[0]) == 0:\n            logger.warning("Empty keep_indices; skip pruning")\n            return\n\n        self._shrink_positions_and_rerotate_keys(keep_indices_all_layers)\n        self._prune_activity_cache(keep_indices_all_layers)\n        \n        logger.info(f"Strict-shrunk KV cache. New length: {get_cache_seq_len(self.kv_cache)}")\n\n    def allocate_budget_by_depth(self, total_budget, num_layers):\n        """Allocate budget uniformly across layers."""\n        budget_per_layer = [total_budget // num_layers] * num_layers\n        diff = total_budget - sum(budget_per_layer)\n        budget_per_layer[-1] += diff\n        return budget_per_layer\n\n    def _compute_attention_scores_manually(self, input_ids, past_key_values):\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len = input_ids.shape[1]\n        batch = input_ids.shape[0]\n        \n        inputs_embeds = self.get_input_embeddings()(input_ids)\n        \n        config = self.language_model.config\n        num_layers = config.num_hidden_layers\n        num_heads = config.num_attention_heads\n        num_key_value_heads = config.num_key_value_heads\n        head_dim = config.hidden_size // num_heads\n        \n        attention_weights_list = []\n        \n        hidden_states = inputs_embeds\n        \n        for layer_idx in range(num_layers):\n            layer = self.language_model.model.layers[layer_idx]\n            \n            past_k, past_v = past_key_values[layer_idx]\n            \n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len, batch)\n            \n            hidden_states_norm = layer.input_layernorm(hidden_states)\n            \n            attn = layer.self_attn\n            \n            query_states = attn.q_proj(hidden_states_norm)\n            query_states = query_states.view(batch, q_len, num_heads, head_dim).transpose(1, 2)\n            \n            key_states = attn.k_proj(hidden_states_norm)\n            key_states = key_states.view(batch, q_len, num_key_value_heads, head_dim).transpose(1, 2)\n            \n            value_states = attn.v_proj(hidden_states_norm)\n            value_states = value_states.view(batch, q_len, num_key_value_heads, head_dim).transpose(1, 2)\n            \n            kv_seq_len = position_ids.max().item() + 1\n            rotary_emb = _get_rotary_module(self.language_model)\n            cos, sin = rotary_emb(value_states, kv_seq_len)\n            cos = cos[position_ids].unsqueeze(1)\n            sin = sin[position_ids].unsqueeze(1)\n            \n            query_states = apply_rotary_pos_emb_1d(query_states, cos, sin)\n            key_states = apply_rotary_pos_emb_1d(key_states, cos, sin)\n            \n            key_states = torch.cat([past_k, key_states], dim=2)\n            value_states = torch.cat([past_v, value_states], dim=2)\n            \n            if num_key_value_heads != num_heads:\n                n_rep = num_heads // num_key_value_heads\n                key_states = torch.repeat_interleave(key_states, n_rep, dim=1)\n                value_states = torch.repeat_interleave(value_states, n_rep, dim=1)\n            \n            attn_weights = torch.matmul(query_states.float(), key_states.float().transpose(-2, -1)) / (head_dim ** 0.5)\n            attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n            attention_weights_list.append(attn_weights)\n        \n        return attention_weights_list\n\n    def prune_kv_cache_by_attention(self, attn_weights_local, attn_weights_global, attn_weights_mixed, num_keep=3000):\n        device = self.device\n        visual_start_idx = self.visual_start_idx\n        num_layers = len(attn_weights_local)\n        \n        curr_cache_lens = self._get_cache_seq_len_per_layer()\n        \n        question_len_local = attn_weights_local[0].shape[2]\n        question_len_global = attn_weights_global[0].shape[2]\n        question_len_mixed = attn_weights_mixed[0].shape[2]\n\n        total_budget = num_keep * num_layers\n        budget_per_layer = self.allocate_budget_by_depth(total_budget, num_layers)\n        \n        keep_indices_all_layers = []\n\n        layer_raw_scores = []\n        layer_configs = []\n\n        for layer_idx in range(len(attn_weights_local)):\n            curr_len = curr_cache_lens[layer_idx]\n            if self.token_activity_cache[layer_idx] is None:\n                self.token_activity_cache[layer_idx] = torch.zeros(curr_len, device=device, dtype=torch.float32)\n            elif self.token_activity_cache[layer_idx].shape[0] < curr_len:\n                diff = curr_len - self.token_activity_cache[layer_idx].shape[0]\n                zeros = torch.zeros(diff, device=device, dtype=torch.float32)\n                self.token_activity_cache[layer_idx] = torch.cat([self.token_activity_cache[layer_idx], zeros])\n            \n            if self.token_activity_cache[layer_idx].shape[0] > curr_len:\n                 self.token_activity_cache[layer_idx] = self.token_activity_cache[layer_idx][:curr_len]\n\n            # Memory hierarchy mode\n            if layer_idx < self.short_term_threshold:\n                layer_type = "short-term"\n                layer_attn_weights = attn_weights_local[layer_idx]\n                question_len = question_len_local\n                layer_recency_alpha = 1\n                k = 20\n                \n            elif layer_idx >= self.long_term_threshold:\n                layer_type = "long-term"\n                layer_attn_weights = attn_weights_global[layer_idx]\n                question_len = question_len_global\n                layer_recency_alpha = 0\n                k = 0.0\n                \n            else:\n                layer_type = "mid-term"\n                layer_attn_weights = attn_weights_mixed[layer_idx]\n                question_len = question_len_mixed\n                progress = (layer_idx - self.short_term_threshold) / (self.long_term_threshold - self.short_term_threshold)\n                layer_recency_alpha = 1.0 - 0.9 * progress\n                k = 20 - 19.5 * progress\n            \n            visual_attn_weights = layer_attn_weights[0].mean(dim=0)[:,visual_start_idx:-1*question_len].mean(dim=0)\n            num_visual_tokens = visual_attn_weights.shape[0]\n            \n            end_idx = visual_start_idx + num_visual_tokens\n            if end_idx <= self.token_activity_cache[layer_idx].shape[0]:\n                self.token_activity_cache[layer_idx][visual_start_idx:end_idx] += visual_attn_weights\n            \n            layer_budget = budget_per_layer[layer_idx]\n            \n            if layer_type == \'long-term\':\n                layer_budget = max(0, layer_budget - 1)\n\n            positions = torch.arange(num_visual_tokens, device=device, dtype=torch.float32)\n            time_distances = (num_visual_tokens - 1 - positions) / max(num_visual_tokens - 1, 1)\n            \n            # Exponential decay\n            recency_weights = torch.exp(-k * time_distances)\n            \n            attn_norm = (visual_attn_weights - visual_attn_weights.min()) / \\\n                        (visual_attn_weights.max() - visual_attn_weights.min() + 1e-6)\n            recency_norm = (recency_weights - recency_weights.min()) / \\\n                        (recency_weights.max() - recency_weights.min() + 1e-6)\n            \n            raw_score = attn_norm * (1 - layer_recency_alpha) + recency_norm * layer_recency_alpha\n            \n            layer_raw_scores.append(raw_score)\n            layer_configs.append({\n                \'budget\': min(layer_budget, num_visual_tokens),\n                \'layer_type\': layer_type,\n                \'visual_start_idx\': visual_start_idx\n            })\n        \n        refined_scores = [s.clone() for s in layer_raw_scores]\n        \n        for i in range(len(refined_scores) - 2, -1, -1):\n            current_type = layer_configs[i][\'layer_type\']\n            \n            # TODO: change hard code here\n            if current_type == \'long-term\':\n                gamma = 0.4  \n            elif current_type == \'mid-term\':\n                gamma = 0.3  \n            else:\n                gamma = 0.1\n            \n            score_current = refined_scores[i]\n            score_next = refined_scores[i+1]\n            \n            if score_current.shape[0] != score_next.shape[0]:\n                score_next_reshaped = score_next.view(1, 1, -1)\n                score_next_interp = F.interpolate(\n                    score_next_reshaped, \n                    size=score_current.shape[0], \n                    mode=\'linear\', \n                    align_corners=False\n                ).view(-1)\n                refined_scores[i] = (1 - gamma) * score_current + gamma * score_next_interp\n            else:\n                refined_scores[i] = (1 - gamma) * score_current + gamma * score_next\n\n        for layer_idx, score in enumerate(refined_scores):\n            config = layer_configs[layer_idx]\n            actual_num_keep = config[\'budget\']\n            start_idx = config[\'visual_start_idx\']\n            \n            topk_indices_relative = torch.topk(score, actual_num_keep, sorted=False)[1]\n            topk_indices_absolute = topk_indices_relative + start_idx\n            topk_indices_absolute_sorted = torch.sort(topk_indices_absolute)[0]\n            \n            keep_indices = torch.cat([\n                torch.arange(start_idx, device=device), \n                topk_indices_absolute_sorted\n            ]).tolist()\n            \n            keep_indices_all_layers.append(keep_indices)\n        \n        return keep_indices_all_layers\n    \n    @torch.inference_mode()\n    def pseudo_forward(self, local_question=None, global_question=None):\n        device = self.device\n        \n        if local_question is None:\n            local_question = "What is happening in the video?"\n        if global_question is None:\n            global_question = "What is the main topic of the video?"\n        \n        local_input_ids = self.processor.tokenizer(local_question).input_ids\n        local_input_ids = torch.as_tensor([local_input_ids], device=device, dtype=torch.int)\n\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len_local = local_input_ids.shape[1]\n        batch = local_input_ids.shape[0]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_local, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_local = self._build_position_ids(start_pos_per_layer[0], q_len_local, batch)\n\n        max_start = max(start_pos_per_layer) if start_pos_per_layer else 0\n        self._set_rotary_required_len(max_start + q_len_local)\n\n        use_flash_attn = (hasattr(self.language_model.config, \'_attn_implementation\') and \n                        self.language_model.config._attn_implementation in ["flash_attention_2", "sdpa"])\n\n        if use_flash_attn:\n            attn_weights_local = self._compute_attention_scores_manually(local_input_ids, self.kv_cache)\n        else:\n            out_local = self.language_model(\n                input_ids=local_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_local,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_local = out_local.attentions\n        \n        global_input_ids = self.processor.tokenizer(global_question).input_ids\n        global_input_ids = torch.as_tensor([global_input_ids], device=device, dtype=torch.int)\n        \n        q_len_global = global_input_ids.shape[1]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_global, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_global = self._build_position_ids(start_pos_per_layer[0], q_len_global, batch)\n        \n        self._set_rotary_required_len(max_start + q_len_global)\n        \n        if use_flash_attn:\n            attn_weights_global = self._compute_attention_scores_manually(global_input_ids, self.kv_cache)\n        else:\n            out_global = self.language_model(\n                input_ids=global_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_global,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_global = out_global.attentions\n        \n        mixed_question = local_question + "; " + global_question\n        mixed_input_ids = self.processor.tokenizer(mixed_question).input_ids\n        mixed_input_ids = torch.as_tensor([mixed_input_ids], device=device, dtype=torch.int)\n        \n        q_len_mixed = mixed_input_ids.shape[1]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_mixed, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_mixed = self._build_position_ids(start_pos_per_layer[0], q_len_mixed, batch)\n        \n        self._set_rotary_required_len(max_start + q_len_mixed)\n        \n        if use_flash_attn:\n            attn_weights_mixed = self._compute_attention_scores_manually(mixed_input_ids, self.kv_cache)\n        else:\n            out_mixed = self.language_model(\n                input_ids=mixed_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_mixed,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_mixed = out_mixed.attentions\n        \n        self._layer_position_ids.clear()\n        \n        print(f"GPU memory usage: {self.get_gpu_memory_usage_gb()} GB")\n        current_k_states_len = self.kv_cache[0][0].shape[2]\n        \n        keep_indices_all_layers = self.prune_kv_cache_by_attention(\n            attn_weights_local, attn_weights_global, attn_weights_mixed, \n            num_keep=self.kv_size\n        )\n        \n        if current_k_states_len > self.kv_size:\n            print(f"Applying KV-Cache compression due to k_states > {self.kv_size}")\n            self.apply_kv_cache_pruning_strict(keep_indices_all_layers)\n    \n    @torch.inference_mode()\n    def predict_and_compress(self):\n        local_question, global_question = self.predict_next_question()\n        self.pseudo_forward(local_question, global_question)\n\n    @torch.inference_mode()\n    def question_answering(self, input_text, max_new_tokens=128, temperature=0, repetition_penalty=1.1, pseudo_forward=False):\n        device = self.device\n        stop_token_ids = [self.processor.tokenizer.eos_token_id]\n        output_ids = []\n\n        start_time = time.perf_counter()\n        \n        prompt = input_text[\'prompt\']\n        input_ids = self.processor.tokenizer(prompt).input_ids\n        input_ids = torch.as_tensor([input_ids], device=device)\n\n        past_lens_prefill = self._get_cache_seq_len_per_layer()\n        start_pos_prefill = self._get_next_start_pos_per_layer()\n\n        inputs_embeds = self.get_input_embeddings()(input_ids)\n        q_len_prefill = inputs_embeds.shape[1]\n        batch = inputs_embeds.shape[0]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_prefill[layer_idx], q_len_prefill, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids = self._build_position_ids(start_pos_prefill[0], q_len_prefill, batch)\n\n        max_start_pos = max(start_pos_prefill) if start_pos_prefill else 0\n        self._set_rotary_required_len(max_start_pos + q_len_prefill)\n\n        out = self.language_model(\n            inputs_embeds=inputs_embeds,\n            use_cache=True,\n            past_key_values=self.kv_cache,\n            position_ids=position_ids,\n            cache_position=None,\n            attention_mask=None,\n        )\n        past_key_values = out.past_key_values\n        logits = out.logits\n        \n        self._append_position_ids(start_pos_prefill, q_len_prefill)\n        self._layer_position_ids.clear()\n\n        for step in range(max_new_tokens):\n            last_token_logits = logits[0, -1, :]\n\n            if repetition_penalty != 1.0 and len(output_ids) > 0:\n                for token_id in set(output_ids):\n                    if last_token_logits[token_id] < 0:\n                        last_token_logits[token_id] *= repetition_penalty\n                    else:\n                        last_token_logits[token_id] /= repetition_penalty\n\n            if temperature == 0.0:\n                _, indices = torch.topk(last_token_logits, 1)\n                token = int(indices[0])\n            else:\n                scaled_logits = last_token_logits / temperature\n                scaled_logits = torch.nan_to_num(\n                    scaled_logits, nan=-float(\'inf\'), posinf=float(\'inf\'), neginf=-float(\'inf\')\n                )\n                probs = F.softmax(scaled_logits, dim=-1)\n                probs = torch.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)\n                probs_sum = probs.sum()\n                if probs_sum > 0:\n                    probs = probs / probs_sum\n                    token = torch.multinomial(probs, num_samples=1).item()\n                else:\n                    _, indices = torch.topk(last_token_logits, 1)\n                    token = int(indices[0])\n\n            output_ids.append(token)\n            if token in stop_token_ids:\n                break\n\n            curr_start_pos = self._get_next_start_pos_per_layer()\n            \n            max_curr_start = max(curr_start_pos) if curr_start_pos else 0\n            self._set_rotary_required_len(max_curr_start + 1)\n            \n            self._layer_position_ids.clear()\n            for layer_idx in range(self.num_layers):\n                pos_step = torch.tensor([curr_start_pos[layer_idx]], device=device, dtype=torch.long)\n                position_ids_layer = pos_step.view(1, 1)\n                self._layer_position_ids[layer_idx] = position_ids_layer\n            \n            pos_step = torch.tensor([curr_start_pos[0]], device=device, dtype=torch.long)\n            position_ids = pos_step.view(1, 1)\n\n            out = self.language_model(\n                input_ids=torch.as_tensor([[token]], device=device),\n                use_cache=True,\n                past_key_values=past_key_values,\n                position_ids=position_ids,\n                cache_position=None,\n                attention_mask=None,\n            )\n            \n            if (not pseudo_forward) and (step == 0):\n                end_time = time.perf_counter()\n                print(f"TTFT: {end_time - start_time} seconds")\n                \n            logits = out.logits\n            past_key_values = out.past_key_values\n\n            self._append_position_ids(curr_start_pos, 1)\n            self._layer_position_ids.clear()\n\n        output = self.processor.tokenizer.decode(\n            output_ids,\n            skip_special_tokens=True,\n            spaces_between_special_tokens=False,\n            clean_up_tokenization_spaces=True,\n        )\n\n        if not pseudo_forward:\n            current_question = input_text[\'question\']\n            current_options = None\n            formatted_question = input_text.get(\'formatted_question\', None)\n            if formatted_question:\n                option_matches = re.findall(r\'\\([A-Z]\\)\\s*(.+?)(?=\\n\\([A-Z]\\)|\\nThe best answer|\\n*$)\', formatted_question, re.DOTALL)\n                if option_matches:\n                    current_options = [opt.strip() for opt in option_matches]\n            self.conv_history.append((current_question, output, current_options))\n            logger.info(f"Saved conversation to history. Total conversations: {len(self.conv_history)}")\n\n        self._truncate_kv_cache(past_lens_prefill)\n        for layer_idx in range(self.num_layers):\n            if (self._position_ids_cache[layer_idx] is not None and \n                self._position_ids_cache[layer_idx].numel() > past_lens_prefill[layer_idx]):\n                self._position_ids_cache[layer_idx] = self._position_ids_cache[layer_idx][\n                    :past_lens_prefill[layer_idx]\n                ].contiguous()\n\n        new_lens = self._get_cache_seq_len_per_layer()\n        print(f"Answering Cache lengths: min={min(new_lens)}, max={max(new_lens)}")\n        return output\n    \n    def _truncate_kv_cache(self, target_lengths):\n        if self.kv_cache is None:\n            return\n        \n        truncated_cache = []\n        for layer_idx, (k_cache, v_cache) in enumerate(self.kv_cache):\n            if isinstance(target_lengths, int):\n                target_len = target_lengths\n            else:\n                target_len = target_lengths[layer_idx]\n            \n            truncated_k = k_cache[:, :, :target_len, :]\n            truncated_v = v_cache[:, :, :target_len, :]\n            truncated_cache.append((truncated_k, truncated_v))\n        \n        if isinstance(self.kv_cache, DynamicCache):\n            self.kv_cache = DynamicCache.from_legacy_cache(truncated_cache)\n        else:\n            self.kv_cache = truncated_cache\n\n\ndef load_model(model_path=\'llava-onevision-qwen2-7b-ov-hf\',\n               n_init=None, kv_size=None, streaming=True, device="cuda", sample_fps=0.5):\n    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)\n    processor.tokenizer.padding_side = \'left\'\n    \n    system_prompt = \'<|im_start|>system \\nYou are a helpful assistant.<|im_end|><|im_start|>user \'\n    init_prompt_ids = processor.tokenizer(system_prompt, return_tensors="pt").input_ids.to(device)\n    \n    base_model = LlavaOnevisionForConditionalGeneration.from_pretrained(\n        model_path,\n        device_map="auto",\n        torch_dtype=torch.float16,\n        trust_remote_code=True,\n    )\n\n    model = LlavaOneVision_Hermes.__new__(LlavaOneVision_Hermes)\n    model.__dict__ = base_model.__dict__.copy()\n\n    Abstract_Hermes.__init__(\n        model, \n        processor, \n        init_prompt_ids.tolist(), \n        kv_size,\n    )\n    model.streaming = streaming\n    \n    num_layers = base_model.language_model.config.num_hidden_layers\n    model.num_layers = num_layers\n    model._position_ids_cache = [None for _ in range(num_layers)]\n    \n    model.short_term_ratio = 0.1\n    model.long_term_ratio = 0.3\n    model.short_term_threshold = int(model.num_layers * model.short_term_ratio)\n    model.long_term_threshold = int(model.num_layers * (1 - model.long_term_ratio))\n    model.token_activity_cache = [None for _ in range(num_layers)]\n    \n    model.total_processed_frames = 0\n    \n    model._layer_position_ids = {}\n    model._hook_handles = []\n    model._register_forward_hooks()\n    model._patch_rotary_embeddings()\n    \n    logger.info(f\'n_init: {init_prompt_ids.shape[1] if n_init is None else n_init}\')\n    logger.info(f\'kv_size: {kv_size}\')\n\n    model.eval()\n\n    return model, processor\n', 'pinned_source/inference/abstract_hermes.py': 'import torch\n\nclass Abstract_Hermes:\n    kv_cache = None\n\n    def __init__(self, processor, init_prompt_ids, kv_size):\n        self.processor = processor\n        self.init_prompt_ids = init_prompt_ids\n        self.kv_size = kv_size\n        self.last_encoded_frames = 0\n        self.visual_start_idx = 14\n        self.conv_history = []\n        \n    def clear_cache(self):\n        self.kv_cache = None\n        self.conv_history = []\n        torch.cuda.empty_cache()\n        torch.cuda.ipc_collect()\n\n    def encode_init_prompt(self):\n        if not isinstance(self.init_prompt_ids, torch.Tensor):\n            self.init_prompt_ids = torch.as_tensor(self.init_prompt_ids, device=self.device)\n        output = self.language_model(input_ids=self.init_prompt_ids, use_cache=True, return_dict=True)\n        self.kv_cache = output.past_key_values\n        self.visual_start_idx = self.kv_cache[0][0].shape[2]\n\n    def get_prompt(self, query, mc=False):\n        prompt = f"\\n{query}<|im_end|><|im_start|>assistant\\n"\n        \n        if mc:\n            prompt += \'Best option: (\'\n        return prompt\n\n    def get_gpu_memory_usage_gb(self):\n        return torch.cuda.max_memory_allocated() / (1024**3)', 'pinned_source/inference/reindex_1d.py': 'import torch\nfrom typing import Tuple\nfrom transformers import DynamicCache\n\n\ndef get_cache_seq_len(past_key_values) -> int:\n    if past_key_values is None:\n        return 0\n    if isinstance(past_key_values, DynamicCache):\n        return past_key_values.get_seq_length()\n    return past_key_values[0][0].shape[2]\n\n\ndef contiguous_kv(past_key_values):\n    new_legacy = []\n    for (k_layer, v_layer) in past_key_values:\n        new_legacy.append((k_layer.contiguous(), v_layer.contiguous()))\n    return new_legacy\n\n\ndef _get_rotary_module(llm) -> torch.nn.Module:\n    if hasattr(llm, "rotary_emb"):\n        return llm.rotary_emb\n    if hasattr(llm, "model") and hasattr(llm.model, "rotary_emb"):\n        return llm.model.rotary_emb\n\n    if hasattr(llm, "model") and hasattr(llm.model, "layers"):\n        if len(llm.model.layers) > 0 and hasattr(llm.model.layers[0], "self_attn"):\n            if hasattr(llm.model.layers[0].self_attn, "rotary_emb"):\n                return llm.model.layers[0].self_attn.rotary_emb\n    raise AttributeError("Cannot find rotary_emb module on language_model")\n\n\ndef rotate_half(x):\n    x1 = x[..., : x.shape[-1] // 2]\n    x2 = x[..., x.shape[-1] // 2 :]\n    return torch.cat((-x2, x1), dim=-1)\n\n\ndef apply_rotary_pos_emb_1d(x, cos, sin):\n    if cos.dim() == 3 and x.dim() == 4:\n        cos = cos.unsqueeze(1)  # [batch, 1, seq_len, head_dim]\n        sin = sin.unsqueeze(1)  # [batch, 1, seq_len, head_dim]\n    \n    x_embed = (x * cos) + (rotate_half(x) * sin)\n    return x_embed\n\n\ndef rotary_delta(cos_old, sin_old, cos_new, sin_new):\n    cos_delta = cos_new * cos_old + sin_new * sin_old\n    sin_delta = sin_new * cos_old - cos_new * sin_old\n    return cos_delta, sin_delta\n\n\n@torch.inference_mode()\ndef compute_cos_sin_for_positions(\n    llm, \n    seq_len: int, \n    position_ids_1d: torch.Tensor, \n    dtype: torch.dtype, \n    device: torch.device\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    \n    rotary_emb = _get_rotary_module(llm)\n    \n    config = llm.config if hasattr(llm, "config") else llm.model.config\n    hidden_size = getattr(config, "hidden_size", 4096)\n    \n    position_ids = position_ids_1d.view(1, -1).to(device)\n    \n    dummy_h = torch.zeros((1, seq_len, hidden_size), device=device, dtype=dtype)\n    \n    \n    seq_len_rope = position_ids.max().item() + 1\n    cos, sin = rotary_emb(dummy_h, seq_len=seq_len_rope)\n    cos = cos[position_ids]\n    sin = sin[position_ids]\n    \n    cos = cos.to(dtype)\n    sin = sin.to(dtype)\n    \n    return cos, sin\n\n@torch.inference_mode()\ndef compute_cos_sin_with_custom_base(\n    llm,\n    seq_len: int,\n    position_ids_1d: torch.Tensor,\n    dtype: torch.dtype,\n    device: torch.device,\n    rope_base: float = 10000.0\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    \n    config = llm.config if hasattr(llm, "config") else llm.model.config\n    hidden_size = getattr(config, "hidden_size", 4096)\n    num_attention_heads = getattr(config, "num_attention_heads", 32)\n    head_dim = hidden_size // num_attention_heads\n    \n    \n    inv_freq = 1.0 / (rope_base ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim))\n    \n    freqs = torch.outer(position_ids_1d.float(), inv_freq)\n    \n    \n    emb = torch.cat([freqs, freqs], dim=-1)\n    \n    cos = emb.cos().unsqueeze(0).to(dtype)  # [1, seq_len, head_dim]\n    sin = emb.sin().unsqueeze(0).to(dtype)  # [1, seq_len, head_dim]\n    \n    return cos, sin\n\n\ndef get_layer_rope_base(\n    layer_idx: int,\n    num_layers: int,\n    short_term_ratio: float = 0.3,\n    long_term_ratio: float = 0.3,\n    short_term_base: float = 10000.0,\n    mid_term_base_range: Tuple[float, float] = (10000.0, 50000.0),\n    long_term_base: float = 50000.0,\n) -> float:\n    \n    short_term_threshold = int(num_layers * short_term_ratio)\n    long_term_threshold = int(num_layers * (1 - long_term_ratio))\n    \n    if layer_idx < short_term_threshold:\n        \n        return short_term_base\n    elif layer_idx >= long_term_threshold:\n        \n        return long_term_base\n    else:\n        \n        mid_start = short_term_threshold\n        mid_end = long_term_threshold\n        progress = (layer_idx - mid_start) / max(mid_end - mid_start, 1)\n        base_start, base_end = mid_term_base_range\n        return base_start + (base_end - base_start) * progress\n\n\n@torch.inference_mode()\ndef apply_rotary_delta_to_keys_only(\n    key_states: torch.Tensor, \n    cos_delta: torch.Tensor, \n    sin_delta: torch.Tensor\n) -> torch.Tensor:\n    \n    k_rot = apply_rotary_pos_emb_1d(key_states, cos_delta, sin_delta)\n    return k_rot\n\n', 'provenance/twinpast_runner.py': '"""Run the frozen, annotation-guided TwinPast development diagnostic."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport shutil\nimport statistics\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nARTIFACT = "HERMES_twinpast_diagnostic_evidence.zip"\nARMS = ("question_only", "recent_four", "history_a", "history_b")\nORDERS = ("canonical", "reversed")\nLETTERS = "ABCDEFGH"\nHOSTS = ("huggingface.co", "hf-mirror.com")\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef digest(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef atomic(path, data):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef fetch(bundle, relative, dest, expected_sha, size=None, reuse=None):\n    import requests\n\n    if dest.is_file() and digest(dest) == expected_sha:\n        return\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temporary = dest.with_suffix(".part")\n    if reuse and reuse.is_file() and digest(reuse) == expected_sha:\n        shutil.copyfile(reuse, temporary)\n        temporary.replace(dest)\n        print(f"MEDIA_REUSED {relative}", flush=True)\n        return\n    cap = size if size is not None else 2 * 1024**2\n    remaining_attempts = [(host, direct) for direct in (False, True) for host in HOSTS]\n    for host, direct in remaining_attempts:\n        url = f"https://{host}/datasets/{bundle[\'manifest\'][\'dataset\']}/resolve/{bundle[\'manifest\'][\'revision\']}/{relative}"\n        try:\n            with requests.Session() as session:\n                session.trust_env = not direct\n                with session.get(url, stream=True, timeout=(8, 15)) as response, temporary.open("wb") as stream:\n                    response.raise_for_status()\n                    start, last_report, count = time.monotonic(), time.monotonic(), 0\n                    for block in response.iter_content(128 * 1024):\n                        count += len(block)\n                        now = time.monotonic()\n                        if count > cap or now - start > 120:\n                            raise ValueError("Download exceeded size or elapsed-time bound")\n                        stream.write(block)\n                        if now - last_report >= 8:\n                            print(f"MEDIA_PROGRESS {relative} {count / 1024**2:.1f} MiB", flush=True)\n                            last_report = now\n            if (size is not None and count != size) or digest(temporary) != expected_sha:\n                raise ValueError("Downloaded size or SHA-256 does not match")\n            temporary.replace(dest)\n            print(f"MEDIA_READY {relative} {count / 1024**2:.1f} MiB", flush=True)\n            return\n        except (requests.RequestException, OSError, ValueError) as exc:\n            print(f"MEDIA_RETRY {relative} host={host} direct={direct} reason={type(exc).__name__}", flush=True)\n    raise RuntimeError(f"Could not acquire {relative}. Verified completed files remain available for rerun.")\n\n\ndef cases(bundle):\n    plans = {row["pair_id"]: row for row in bundle["manifest"]["selected_pairs"]}\n    visual = {row["pair_id"]: row for row in bundle["visual"]}\n    media = {row["pair_id"]: row for row in bundle["media"]}\n    assert len(plans) == len(visual) == len(media) == 12 and plans.keys() == visual.keys() == media.keys()\n    selected = []\n    for pair_id, plan in plans.items():\n        if visual[pair_id]["status"] == "clear_contrast":\n            checked = media[pair_id]\n            assert checked["status"] == "checked" and checked["task"] == plan["task"]\n            assert checked["grid_s"] == sorted(set(checked["grid_s"])) and len(checked["grid_s"]) == 6\n            assert checked["grid_s"][-4:] == plan["latest_four_grid_s"]\n            assert checked["grid_s"][-1] < plan["query_time_s"]\n            for branch in "ab":\n                assert all(checked["decoded"][branch]["early_samples_inside"])\n            assert checked["decoded"]["a"]["decoded_rgb_sha256"][-4:] == checked["decoded"]["b"]["decoded_rgb_sha256"][-4:]\n            selected.append((plan, checked, visual[pair_id]))\n    assert len(selected) == 9\n    return selected\n\n\ndef annotation_rows(bundle, raw):\n    assert sha(raw) == bundle["manifest"]["annotation_sha256"], "Pinned annotations changed"\n    rows = [json.loads(line) for line in raw.splitlines() if line.strip()]\n    result = {row["pair_id"]: row for row in rows}\n    assert len(rows) == len(result) == bundle["manifest"]["dataset_pairs"]\n    for plan, media, visual in cases(bundle):\n        row = result[plan["pair_id"]]\n        assert all(row[field] == plan[field] for field in ("video_a", "video_b", "task"))\n        assert row["t_query"] == plan["query_time_s"] and row["question"].strip()\n        choices = row["options"]\n        assert 2 <= len(choices) <= len(LETTERS) and all(c.strip() for c in choices)\n        assert len({c.strip().casefold() for c in choices}) == len(choices)\n        assert row["answer_a"] != row["answer_b"]\n        for branch in "ab":\n            assert row[f"answer_{branch}"] in LETTERS[:len(choices)]\n            assert row[f"answer_{branch}"] == visual[f"observed_option_{branch}"]\n            seconds = media["decoded"][branch]["frame_start_s"]\n            assert all(t < row["t_query"] for t in seconds)\n            assert all(any(lo - 1e-6 <= t <= hi + 1e-6\n                           for lo, hi in row["evidence_intervals"][branch.upper()]) for t in seconds[:2])\n    return result\n\n\ndef decode(path, recorded):\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    frames = reader.get_batch(recorded["source_frames"]).asnumpy()\n    assert [sha(frame.tobytes()) for frame in frames] == recorded["decoded_rgb_sha256"], (\n        f"Decoded frames differ from the frozen audit for {path.name}")\n    return frames\n\n\ndef features(model, frames):\n    import torch\n\n    chw = torch.from_numpy(frames).permute(0, 3, 1, 2)\n    pixels = model.processor.video_processor(chw, return_tensors="pt").pixel_values_videos\n    output = model.get_video_features(pixels.to(model.device, model.dtype))\n    assert output.shape[0] == 1 and output.shape[1] % len(frames) == 0\n    return output\n\n\ndef score(model, system, prefix, option_ids, visual, cached=False):\n    import torch\n\n    lm = model.language_model\n    embed = lm.get_input_embeddings()\n    question = embed(prefix)\n    left = system if visual is None else torch.cat((system, visual), dim=1)\n    if cached:\n        initial = lm(inputs_embeds=left, use_cache=True, return_dict=True)\n        output = lm(inputs_embeds=question, past_key_values=initial.past_key_values, use_cache=True, return_dict=True)\n    else:\n        output = lm(inputs_embeds=torch.cat((left, question), dim=1), use_cache=False, return_dict=True)\n    result = output.logits[0, -1, option_ids].float().detach().cpu()\n    assert torch.isfinite(result).all()\n    return result, left.shape[1] + question.shape[1]\n\n\ndef make_prompt(model, question, choices, order, with_video):\n    import torch\n\n    perm = list(range(len(choices)))\n    if order == "reversed":\n        perm.reverse()\n    letters = LETTERS[:len(choices)]\n    context = "Answer the question about the video." if with_video else "Answer the question."\n    options = "\\n".join(f"({letter}) {choices[i]}" for letter, i in zip(letters, perm, strict=True))\n    text = model.get_prompt(f"{context}\\nQuestion: {question}\\nOptions:\\n{options}\\n"\n                            "Reply with only the letter of the best option.") + "Answer letter:\\n"\n    tokenizer = model.processor.tokenizer\n    ids = tokenizer.encode(text, add_special_tokens=False)\n    tokens = []\n    for letter in letters:\n        suffix = tokenizer.encode(letter, add_special_tokens=False)\n        assert len(suffix) == 1 and tokenizer.encode(text + letter, add_special_tokens=False) == ids + suffix\n        tokens.extend(suffix)\n    assert len(set(tokens)) == len(tokens)\n    return torch.tensor([ids], device=model.device), tokens, perm, sha(text.encode("utf-8"))\n\n\ndef evaluate_pair(model, plan, media, row, paths, preflight=False):\n    import numpy as np\n    import torch\n\n    frames = {branch: decode(paths[branch], media["decoded"][branch]) for branch in "ab"}\n    assert np.array_equal(frames["a"][2:], frames["b"][2:])\n    recent = features(model, frames["a"][2:])\n    a = features(model, frames["a"][:2])\n    b = features(model, frames["b"][:2])\n    assert a.shape == b.shape and a.shape[1] * 2 == recent.shape[1]\n    visuals = {"question_only": None, "recent_four": recent,\n               "history_a": torch.cat((a, recent), dim=1), "history_b": torch.cat((b, recent), dim=1)}\n    lm = model.language_model\n    system = lm.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    results, checks = [], []\n    for order in ORDERS:\n        prompt = make_prompt(model, row["question"], row["options"], order, True)\n        for arm in ARMS:\n            ids, option_ids, perm, prompt_sha = prompt if arm != "question_only" else make_prompt(\n                model, row["question"], row["options"], order, False)\n            started = time.perf_counter()\n            output, count = score(model, system, ids, option_ids, visuals[arm])\n            elapsed = time.perf_counter() - started\n            if preflight:\n                replay, replay_count = score(model, system, ids, option_ids, visuals[arm])\n                torch.testing.assert_close(output, replay, atol=0, rtol=0)\n                assert replay_count == count\n                if arm == "history_a":\n                    cached, _ = score(model, system, ids, option_ids, visuals[arm], cached=True)\n                    torch.testing.assert_close(output, cached, atol=0.025, rtol=0.005)\n                    assert output.argmax().item() == cached.argmax().item(), "Native-cache top option differs"\n                    checks.append(dict(order=order, native_cache_max_abs=float((output - cached).abs().max())))\n            values = [0.0] * len(perm)\n            for idx, value in zip(perm, output.tolist(), strict=True):\n                values[idx] = value\n            # Canonical tie-breaking prevents label order from deciding exact ties.\n            pred = LETTERS[max(range(len(values)), key=values.__getitem__)]\n            recent_sha = media["decoded"]["a"]["decoded_rgb_sha256"][-4:] if arm != "question_only" else []\n            earlier_sha = media["decoded"][arm[-1]]["decoded_rgb_sha256"][:2] if arm.startswith("history") else []\n            results.append(dict(pair_id=plan["pair_id"], task=plan["task"], order=order, arm=arm, prediction=pred,\n                                gold_a=row["answer_a"], gold_b=row["answer_b"], scores=values, permutation=perm,\n                                label_token_ids=option_ids, prompt_sha256=prompt_sha, lm_tokens=count,\n                                earlier_frame_sha256=earlier_sha, recent_frame_sha256=recent_sha,\n                                forward_s=elapsed))\n    return results, checks\n\n\ndef summarize(blocks):\n    pairs = []\n    for block in blocks:\n        records = {(r["order"], r["arm"]): r for r in block}\n        pair = dict(pair_id=block[0]["pair_id"], task=block[0]["task"])\n        for order in ORDERS:\n            a, b = (records[order, arm] for arm in ("history_a", "history_b"))\n            assert a["lm_tokens"] == b["lm_tokens"] and a["prompt_sha256"] == b["prompt_sha256"]\n            assert a["recent_frame_sha256"] == b["recent_frame_sha256"]\n            ai, bi = (LETTERS.index(a[f"gold_{branch}"]) for branch in "ab")\n            ma = a["scores"][ai] - a["scores"][bi]\n            mb = b["scores"][ai] - b["scores"][bi]\n            wins = int(a["prediction"] == a["gold_a"]) + int(b["prediction"] == b["gold_b"])\n            reversal = a["prediction"] != b["prediction"]\n            pair[order] = dict(correct_branches=wins, correct_reversal=wins == 2, unchanged=not reversal,\n                               wrong_reversal=reversal and wins != 2, margin_a=ma, margin_b=mb,\n                               directional_change=ma - mb)\n            for arm in ("question_only", "recent_four"):\n                prediction = records[order, arm]["prediction"]\n                pair[order][f"{arm}_correct_branches"] = int(prediction in (a["gold_a"], a["gold_b"]))\n        pair["both_orders_correct"] = all(pair[order]["correct_reversal"] for order in ORDERS)\n        pair["predictions_order_stable"] = {\n            arm: records["canonical", arm]["prediction"] == records["reversed", arm]["prediction"] for arm in ARMS\n        }\n        pairs.append(pair)\n    orders = {order: dict(\n        both_correct=sum(row[order]["correct_reversal"] for row in pairs),\n        correct_branches=sum(row[order]["correct_branches"] for row in pairs),\n        question_only_correct_branches=sum(row[order]["question_only_correct_branches"] for row in pairs),\n        recent_four_correct_branches=sum(row[order]["recent_four_correct_branches"] for row in pairs),\n        wrong_reversals=sum(row[order]["wrong_reversal"] for row in pairs),\n        unchanged=sum(row[order]["unchanged"] for row in pairs),\n        median_directional_change=statistics.median(row[order]["directional_change"] for row in pairs))\n        for order in ORDERS} if pairs else {}\n    robust = sum(row["both_orders_correct"] for row in pairs)\n    numeric_gate = len(pairs) == 9 and robust >= 5 and all(x["median_directional_change"] > 0 for x in orders.values())\n    return dict(pairs=len(pairs), robust_correct_pairs=robust, order_summary=orders, per_pair=pairs,\n                numeric_development_gate=numeric_gate,\n                advancement="pending source-group verification" if numeric_gate else\n                ("stop on this backbone if diagnostic validity holds" if len(pairs) == 9 and robust == 0\n                 else "review diagnostic outcomes before advancing"),\n                source_groups="The pinned annotations do not identify original source groups. Verify independently.")\n\n\ndef package(work):\n    artifact = work / ARTIFACT\n    temporary = work / (ARTIFACT + ".tmp")\n    files = [path for name in ("exports", "checkpoints", "provenance", "pinned_source", "logs")\n             for path in (work / name).rglob("*") if path.is_file() and path.suffix != ".tmp"]\n    files += [path for path in (work / "launch.json", work / "completion.json") if path.exists()]\n    with ZipFile(temporary, "w", ZIP_DEFLATED) as archive:\n        entries = []\n        for path in sorted(files):\n            raw = path.read_bytes()\n            relative = path.relative_to(work).as_posix()\n            archive.writestr(relative, raw)\n            entries.append(dict(path=relative, bytes=len(raw), sha256=sha(raw)))\n        archive.writestr("evidence_manifest.json", json.dumps(entries, indent=2))\n    temporary.replace(artifact)\n    print(f"EVIDENCE_ZIP={artifact}", flush=True)\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    bundle_path = work / "provenance" / "bundle.json"\n    bundle = json.loads(bundle_path.read_text(encoding="utf-8"))\n    selection = cases(bundle)\n    dataset = work / "media" / "data.jsonl"\n    fetch(bundle, "data.jsonl", dataset, bundle["manifest"]["annotation_sha256"])\n    rows = annotation_rows(bundle, dataset.read_bytes())\n    atomic(work / "provenance" / "selected_annotations.json", [rows[plan["pair_id"]] for plan, _, _ in selection])\n    needed = sum(meta["bytes"] for plan, media, _ in selection\n                 for branch, meta in media["decoded"].items()\n                 if not (work / "media" / plan[f"video_{branch}"]).is_file()\n                 or (work / "media" / plan[f"video_{branch}"]).stat().st_size != meta["bytes"])\n    assert shutil.disk_usage(work).free >= needed + 256 * 1024**2, "Free data-disk space before running"\n    locations = {}\n    print(f"MEDIA_PLAN 18 pinned clips, up to {needed / 1024**2:.1f} MiB not present", flush=True)\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        pending = []\n        for plan, media, _ in selection:\n            locations[plan["pair_id"]] = {}\n            for branch, meta in media["decoded"].items():\n                relative = plan[f"video_{branch}"]\n                dest = work / "media" / relative\n                reuse = args.media_reuse / Path(relative).relative_to("videos")\n                pending.append(pool.submit(fetch, bundle, relative, dest, meta["sha256"], meta["bytes"], reuse))\n                locations[plan["pair_id"]][branch] = dest\n        for download in pending:\n            download.result()\n    settings = dict(bundle_sha256=digest(bundle_path), runner_sha256=digest(Path(__file__)),\n                    model_name=args.model.name, torch_version=torch.__version__,\n                    gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,\n                    driver=torch.cuda.driver_version() if hasattr(torch.cuda, "driver_version") else None)\n    identity = sha(json.dumps(settings, sort_keys=True).encode())\n    checkpoint_dir = work / "checkpoints"\n    checkpoint_dir.mkdir(exist_ok=True)\n    atomic(work / "exports" / "run_config.json", settings)\n    print("LOADING_PINNED_MODEL", args.model.name, flush=True)\n    model, _ = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda"\n    blocks = []\n    with torch.inference_mode():\n        for index, (plan, media, _) in enumerate(selection):\n            pair_id = plan["pair_id"]\n            checkpoint = checkpoint_dir / f"{pair_id}.json"\n            if index and checkpoint.is_file():\n                saved = json.loads(checkpoint.read_text(encoding="utf-8"))\n                assert saved["identity"] == identity and saved["pair_id"] == pair_id, "Use a new WORK for changed settings"\n                records = saved["records"]\n                assert len(records) == 8 and {(r["order"], r["arm"]) for r in records} == {\n                    (order, arm) for order in ORDERS for arm in ARMS}\n                assert all(r["pair_id"] == pair_id for r in records)\n                print(f"PAIR_REUSED {index + 1}/9 {pair_id}", flush=True)\n            else:\n                print(f"PAIR_RUNNING {index + 1}/9 {pair_id}", flush=True)\n                records, checks = evaluate_pair(model, plan, media, rows[pair_id], locations[pair_id], preflight=index == 0)\n                if index == 0:\n                    if checkpoint.exists():\n                        assert json.loads(checkpoint.read_text(encoding="utf-8"))["identity"] == identity, (\n                            "Use a new WORK for changed settings")\n                    atomic(work / "exports" / "preflight.json", dict(status="pass", pair_id=pair_id, checks=checks))\n                    print("TWINPAST_PREFLIGHT_PASS", flush=True)\n                atomic(checkpoint, dict(identity=identity, pair_id=pair_id, records=records))\n                print(f"PAIR_DONE {index + 1}/9 {pair_id}", flush=True)\n            blocks.append(records)\n    summary = summarize(blocks)\n    atomic(work / "exports" / "summary.json", summary)\n    fields = ("pair_id", "task", "order", "arm", "prediction", "gold_a", "gold_b", "scores", "permutation",\n              "label_token_ids", "prompt_sha256", "lm_tokens", "earlier_frame_sha256", "recent_frame_sha256", "forward_s")\n    with (work / "exports" / "scores.csv").open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows({key: json.dumps(value) if isinstance(value, list) else value for key, value in row.items()}\n                         for block in blocks for row in block)\n    atomic(work / "completion.json", dict(status="complete", pairs=len(blocks), scores=sum(map(len, blocks))))\n    print(f"TWINPAST_DIAGNOSTIC_COMPLETE pairs={len(blocks)} scores={sum(map(len, blocks))}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--work", required=True, type=Path)\n    parser.add_argument("--model", required=True, type=Path)\n    parser.add_argument("--media-reuse", type=Path, default=Path("/root/autodl-tmp/twinpast-development"))\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/bundle.json': '{\n  "manifest": {\n    "dataset": "younght/TwinPast",\n    "revision": "c1f1cbab68bfda2982565c0491f1a97474b73558",\n    "annotation_sha256": "f36073ba77565748175741a0e3a11e0bf6536559d6b002ffa43a87016f30542e",\n    "seed": "HERMES-TwinPast-development-2026-11-23",\n    "purpose": "development_only_exclude_these_pairs_from_confirmation",\n    "dataset_pairs": 350,\n    "pairs_with_evidence_intervals": 289,\n    "eligible_by_task": {\n      "HSR": 12,\n      "AIR": 54\n    },\n    "selected_pairs": [\n      {\n        "pair_id": "twinpast_0208",\n        "task": "HSR",\n        "video_a": "videos/twinpast_0208/A.mp4",\n        "video_b": "videos/twinpast_0208/B.mp4",\n        "query_time_s": 33.958333333333336,\n        "common_earlier_grid_s": [\n          0,\n          2\n        ],\n        "latest_four_grid_s": [\n          26,\n          28,\n          30,\n          32\n        ]\n      },\n      {\n        "pair_id": "twinpast_0094",\n        "task": "HSR",\n        "video_a": "videos/twinpast_0094/A.mp4",\n        "video_b": "videos/twinpast_0094/B.mp4",\n        "query_time_s": 54.125,\n        "common_earlier_grid_s": [\n          6,\n          8,\n          10,\n          12,\n          14\n        ],\n        "latest_four_grid_s": [\n          48,\n          50,\n          52,\n          54\n        ]\n      },\n      {\n        "pair_id": "twinpast_0099",\n        "task": "HSR",\n        "video_a": "videos/twinpast_0099/A.mp4",\n        "video_b": "videos/twinpast_0099/B.mp4",\n        "query_time_s": 52.0,\n        "common_earlier_grid_s": [\n          0,\n          2,\n          4\n        ],\n        "latest_four_grid_s": [\n          44,\n          46,\n          48,\n          50\n        ]\n      },\n      {\n        "pair_id": "twinpast_0252",\n        "task": "HSR",\n        "video_a": "videos/twinpast_0252/A.mp4",\n        "video_b": "videos/twinpast_0252/B.mp4",\n        "query_time_s": 114.08333333333333,\n        "common_earlier_grid_s": [\n          56,\n          58,\n          60\n        ],\n        "latest_four_grid_s": [\n          108,\n          110,\n          112,\n          114\n        ]\n      },\n      {\n        "pair_id": "twinpast_0095",\n        "task": "HSR",\n        "video_a": "videos/twinpast_0095/A.mp4",\n        "video_b": "videos/twinpast_0095/B.mp4",\n        "query_time_s": 42.458333333333336,\n        "common_earlier_grid_s": [\n          2,\n          8\n        ],\n        "latest_four_grid_s": [\n          36,\n          38,\n          40,\n          42\n        ]\n      },\n      {\n        "pair_id": "twinpast_0259",\n        "task": "HSR",\n        "video_a": "videos/twinpast_0259/A.mp4",\n        "video_b": "videos/twinpast_0259/B.mp4",\n        "query_time_s": 37.666666666666664,\n        "common_earlier_grid_s": [\n          22,\n          24,\n          26\n        ],\n        "latest_four_grid_s": [\n          30,\n          32,\n          34,\n          36\n        ]\n      },\n      {\n        "pair_id": "twinpast_0196",\n        "task": "AIR",\n        "video_a": "videos/twinpast_0196/A.mp4",\n        "video_b": "videos/twinpast_0196/B.mp4",\n        "query_time_s": 10.0,\n        "common_earlier_grid_s": [\n          0,\n          2\n        ],\n        "latest_four_grid_s": [\n          2,\n          4,\n          6,\n          8\n        ]\n      },\n      {\n        "pair_id": "twinpast_0185",\n        "task": "AIR",\n        "video_a": "videos/twinpast_0185/A.mp4",\n        "video_b": "videos/twinpast_0185/B.mp4",\n        "query_time_s": 16.958333333333332,\n        "common_earlier_grid_s": [\n          0,\n          2,\n          4\n        ],\n        "latest_four_grid_s": [\n          10,\n          12,\n          14,\n          16\n        ]\n      },\n      {\n        "pair_id": "twinpast_0214",\n        "task": "AIR",\n        "video_a": "videos/twinpast_0214/A.mp4",\n        "video_b": "videos/twinpast_0214/B.mp4",\n        "query_time_s": 44.041666666666664,\n        "common_earlier_grid_s": [\n          0,\n          2,\n          4\n        ],\n        "latest_four_grid_s": [\n          38,\n          40,\n          42,\n          44\n        ]\n      },\n      {\n        "pair_id": "twinpast_0079",\n        "task": "AIR",\n        "video_a": "videos/twinpast_0079/A.mp4",\n        "video_b": "videos/twinpast_0079/B.mp4",\n        "query_time_s": 84.95833333333333,\n        "common_earlier_grid_s": [\n          6,\n          8\n        ],\n        "latest_four_grid_s": [\n          78,\n          80,\n          82,\n          84\n        ]\n      },\n      {\n        "pair_id": "twinpast_0236",\n        "task": "AIR",\n        "video_a": "videos/twinpast_0236/A.mp4",\n        "video_b": "videos/twinpast_0236/B.mp4",\n        "query_time_s": 39.958333333333336,\n        "common_earlier_grid_s": [\n          0,\n          2,\n          4\n        ],\n        "latest_four_grid_s": [\n          32,\n          34,\n          36,\n          38\n        ]\n      },\n      {\n        "pair_id": "twinpast_0036",\n        "task": "AIR",\n        "video_a": "videos/twinpast_0036/A.mp4",\n        "video_b": "videos/twinpast_0036/B.mp4",\n        "query_time_s": 43.958333333333336,\n        "common_earlier_grid_s": [\n          0,\n          2,\n          4\n        ],\n        "latest_four_grid_s": [\n          36,\n          38,\n          40,\n          42\n        ]\n      }\n    ]\n  },\n  "visual": [\n    {\n      "pair_id": "twinpast_0208",\n      "status": "clear_contrast",\n      "observed_option_a": "D",\n      "observed_option_b": "C",\n      "note": "Book moves from closed to open versus from open to closed."\n    },\n    {\n      "pair_id": "twinpast_0094",\n      "status": "clear_contrast",\n      "observed_option_a": "A",\n      "observed_option_b": "B",\n      "note": "Liquid streams from the tilted bottle versus a person holding the bottle below."\n    },\n    {\n      "pair_id": "twinpast_0099",\n      "status": "uncertain",\n      "observed_option_a": "",\n      "observed_option_b": "",\n      "note": "Colored letters are visible, but order in the oblique view is uncertain at this sheet resolution."\n    },\n    {\n      "pair_id": "twinpast_0252",\n      "status": "uncertain",\n      "observed_option_a": "",\n      "observed_option_b": "",\n      "note": "Door position differs at frame 56 s, but both look open at 58 s and the last-attempt endpoint is unclear."\n    },\n    {\n      "pair_id": "twinpast_0095",\n      "status": "clear_contrast",\n      "observed_option_a": "B",\n      "observed_option_b": "A",\n      "note": "Door reaches closed position versus remaining open with stool in the way."\n    },\n    {\n      "pair_id": "twinpast_0259",\n      "status": "clear_contrast",\n      "observed_option_a": "B",\n      "observed_option_b": "A",\n      "note": "Box ends closed versus remaining propped open."\n    },\n    {\n      "pair_id": "twinpast_0196",\n      "status": "insufficient_disjoint_earlier_slots",\n      "observed_option_a": "",\n      "observed_option_b": "",\n      "note": "One older slot overlaps the latest-four window."\n    },\n    {\n      "pair_id": "twinpast_0185",\n      "status": "clear_contrast",\n      "observed_option_a": "A",\n      "observed_option_b": "B",\n      "note": "Bright green nails holding the pen versus no visible green nail polish."\n    },\n    {\n      "pair_id": "twinpast_0214",\n      "status": "clear_contrast",\n      "observed_option_a": "B",\n      "observed_option_b": "A",\n      "note": "Planting bed lacks the large white rabbit ornament versus a clearly visible ornament."\n    },\n    {\n      "pair_id": "twinpast_0079",\n      "status": "clear_contrast",\n      "observed_option_a": "B",\n      "observed_option_b": "A",\n      "note": "Knitted beanie versus a hat with antlers."\n    },\n    {\n      "pair_id": "twinpast_0236",\n      "status": "clear_contrast",\n      "observed_option_a": "B",\n      "observed_option_b": "A",\n      "note": "White robot to the right of monitor versus a black tower."\n    },\n    {\n      "pair_id": "twinpast_0036",\n      "status": "clear_contrast",\n      "observed_option_a": "B",\n      "observed_option_b": "A",\n      "note": "White cup with brown liquid versus brown cup with dark lumps."\n    }\n  ],\n  "media": [\n    {\n      "pair_id": "twinpast_0208",\n      "task": "HSR",\n      "status": "checked",\n      "grid_s": [\n        0,\n        2,\n        26,\n        28,\n        30,\n        32\n      ],\n      "normalized_rgb_difference": [\n        0.07562276839748246,\n        0.07629130355846042,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 19355906,\n          "sha256": "913c198fc475031b0431c84f05332c144925e30ae812ea1b7c4aabafcb81bd7b",\n          "source_frames": [\n            0,\n            48,\n            624,\n            672,\n            720,\n            768\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            26.0,\n            28.0,\n            30.0,\n            32.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "d17e1732e88474db59ecaa97ae569059e3a24aa005105a9798bcc232c1cd0302",\n            "11ef8a873cd1e1bed978562906aba096b3e3679085dccb7b13ae16ee9961c67e",\n            "5cec797760bbb48ec2b000850f628f9a8c8a6102d82926dd7dcbad503703e6d1",\n            "7e2bd866391a4a680cff36d7378410fb2eb2f3fa40c80d9ff560bff9742df0e9",\n            "256582e1e57616173390d7637df3e1a932e7d95f10559c37f04c7013cb5ed7e1",\n            "f572ccd4b3a45967db488bbe4bb2d4ab8b52729bdd709801df63cd205ab00551"\n          ]\n        },\n        "b": {\n          "bytes": 19367971,\n          "sha256": "90da807127daed7dee910a83e831c8d468cc530754e945cc6c2011847812a102",\n          "source_frames": [\n            0,\n            48,\n            624,\n            672,\n            720,\n            768\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            26.0,\n            28.0,\n            30.0,\n            32.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "bbec0ed7398d8620aae53183b9f1a7cc63ef584464991ee24579c1ad43f43333",\n            "af0aeaedd2eaf24ec6d3d3b37a484b580073dfba1f2adfacf5c6bcf968728c07",\n            "5cec797760bbb48ec2b000850f628f9a8c8a6102d82926dd7dcbad503703e6d1",\n            "7e2bd866391a4a680cff36d7378410fb2eb2f3fa40c80d9ff560bff9742df0e9",\n            "256582e1e57616173390d7637df3e1a932e7d95f10559c37f04c7013cb5ed7e1",\n            "f572ccd4b3a45967db488bbe4bb2d4ab8b52729bdd709801df63cd205ab00551"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0094",\n      "task": "HSR",\n      "status": "checked",\n      "grid_s": [\n        6,\n        8,\n        48,\n        50,\n        52,\n        54\n      ],\n      "normalized_rgb_difference": [\n        0.1399284122085048,\n        0.14136489600984428,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 23523559,\n          "sha256": "a4bc3db1a72cf2aa05deacedbcc79a25d60a9bb1e7181886600e08fc6dea4b89",\n          "source_frames": [\n            144,\n            192,\n            1152,\n            1200,\n            1248,\n            1296\n          ],\n          "frame_start_s": [\n            6.0,\n            8.0,\n            48.0,\n            50.0,\n            52.0,\n            54.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "2df65a127e44a6cce55dffd8e071a04dc8ac5040ccb27a70120c64ec7ec8e91c",\n            "727c3cf00a49fa638de8123d168aed84d33a72354687c3b63ce770ed037b3d6b",\n            "e3af61406bb5b76df88ed4daeb586e3cfda60aec7e6f4e8bc0b33ae1896d3a80",\n            "919b30f1b67bba58602603d2b05b3b26b7db17013243ad09e3d990a560baabf7",\n            "91e8ddfd077d7f284e8ec61dac2a41b11d168797815fbc583c05a0de88ff156c",\n            "f7818d6ed63a7576578e5849a81ce7b0ad36aebb8b114061cf2be98b8e8478c5"\n          ]\n        },\n        "b": {\n          "bytes": 22992602,\n          "sha256": "c78deb69dd5f496a65a527262fd48b9aa7e89821efd604893dcbd78984e1bc8c",\n          "source_frames": [\n            144,\n            192,\n            1152,\n            1200,\n            1248,\n            1296\n          ],\n          "frame_start_s": [\n            6.0,\n            8.0,\n            48.0,\n            50.0,\n            52.0,\n            54.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "eb1b798b497684d9605a5def7222f9d0c3602f6708e13247118bb4cfca68444e",\n            "f88bef77ab3ef8170b0542ec2fc6c8722e5aa5414aaddbcdba10cd770ca704a9",\n            "e3af61406bb5b76df88ed4daeb586e3cfda60aec7e6f4e8bc0b33ae1896d3a80",\n            "919b30f1b67bba58602603d2b05b3b26b7db17013243ad09e3d990a560baabf7",\n            "91e8ddfd077d7f284e8ec61dac2a41b11d168797815fbc583c05a0de88ff156c",\n            "f7818d6ed63a7576578e5849a81ce7b0ad36aebb8b114061cf2be98b8e8478c5"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0099",\n      "task": "HSR",\n      "status": "checked",\n      "grid_s": [\n        0,\n        2,\n        44,\n        46,\n        48,\n        50\n      ],\n      "normalized_rgb_difference": [\n        0.23414232076575486,\n        0.23253613430969094,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 27571576,\n          "sha256": "3612dacef08616a655a52b2cb80d729167adc6f6f1058ef0a76728a4bb07a167",\n          "source_frames": [\n            0,\n            48,\n            1056,\n            1104,\n            1152,\n            1200\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            44.0,\n            46.0,\n            48.0,\n            50.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "a47938e1c69ebdb5f06f70a98b7d43bb83c891d3d03bc3cb7099bcfd38405efc",\n            "40b486bc71a199242d3d4ae21eea9ec44a5090da0172f3a2293b82f827551255",\n            "72df65e3d50cff007e55f9a23b32053528d4ec76e2ae21f519ac6846b580165c",\n            "d8acd3a595ac474ed235509f5c3f5654bcdb934e10706544186e366423bc76c3",\n            "5b27abce92fcde01ca059eeb30a665acbf54151fc82bc1a396e5d245c278e85f",\n            "e4365f9aadb90c2570cc0a86a32641055e94fe59fc361cf80101faa81fb4aa11"\n          ]\n        },\n        "b": {\n          "bytes": 29849686,\n          "sha256": "6ea5bb359efb0113e352a0e2ca3f0715a45fd088a3a411561606634e859f1639",\n          "source_frames": [\n            0,\n            48,\n            1056,\n            1104,\n            1152,\n            1200\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            44.0,\n            46.0,\n            48.0,\n            50.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "3bc6902d2a0683b43c34da2269dd443e2582c1e717dfb33670a1d8ba56075b16",\n            "8c00c9cbdd52ec8903497bfdf203f35ac118529a7b287f9e1f21d8d4e5fec426",\n            "72df65e3d50cff007e55f9a23b32053528d4ec76e2ae21f519ac6846b580165c",\n            "d8acd3a595ac474ed235509f5c3f5654bcdb934e10706544186e366423bc76c3",\n            "5b27abce92fcde01ca059eeb30a665acbf54151fc82bc1a396e5d245c278e85f",\n            "e4365f9aadb90c2570cc0a86a32641055e94fe59fc361cf80101faa81fb4aa11"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0252",\n      "task": "HSR",\n      "status": "checked",\n      "grid_s": [\n        56,\n        58,\n        108,\n        110,\n        112,\n        114\n      ],\n      "normalized_rgb_difference": [\n        0.10871535342532075,\n        0.10234652374324216,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 38872585,\n          "sha256": "903f8e8ac3f39c4b504db4836d23631fa41e401642ada4fa115ee0fa4f2f1e73",\n          "source_frames": [\n            1344,\n            1392,\n            2592,\n            2640,\n            2688,\n            2736\n          ],\n          "frame_start_s": [\n            56.0,\n            58.0,\n            108.0,\n            110.0,\n            112.0,\n            114.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "a9d3c94531e27053e6880aa93e93db68427d3ec79983be0e7c03c3c97c897e3a",\n            "128812291e83b528d1a9dad4b4afc7ebc0141e84cf493f1458823d37782b2d19",\n            "cfe7d5c94c62e9bd737096dd9e14580fcdb93363c7b41719c6f7f50ad3e2a2c1",\n            "0b7a5a775f37dea6e4b14b9ef27820f9eb1f165c22a22b7656e3f4e1cf6d67cc",\n            "a8e0b7862c86d3788a3aef828c1b1c88178c1b1b2d4077ee646e62e7b82e11f1",\n            "6bd362f5718cacdc6a5486bf4b9dd74d0a3547e40e4e98b04476b233e2e503e9"\n          ]\n        },\n        "b": {\n          "bytes": 39208455,\n          "sha256": "df5ec6d9f88971a2a6bf4a93bd5a8ff0be21b6aef15e8af1f7869abed3c692d5",\n          "source_frames": [\n            1344,\n            1392,\n            2592,\n            2640,\n            2688,\n            2736\n          ],\n          "frame_start_s": [\n            56.0,\n            58.0,\n            108.0,\n            110.0,\n            112.0,\n            114.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "4110f6990dd937353b42dac6eaa56b721a761e997946be41de23a699971da4d4",\n            "8a8d8099706efe2053bc83b55765dcf6b56e15ee7b8064faa0ffa4f5220af935",\n            "cfe7d5c94c62e9bd737096dd9e14580fcdb93363c7b41719c6f7f50ad3e2a2c1",\n            "0b7a5a775f37dea6e4b14b9ef27820f9eb1f165c22a22b7656e3f4e1cf6d67cc",\n            "a8e0b7862c86d3788a3aef828c1b1c88178c1b1b2d4077ee646e62e7b82e11f1",\n            "6bd362f5718cacdc6a5486bf4b9dd74d0a3547e40e4e98b04476b233e2e503e9"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0095",\n      "task": "HSR",\n      "status": "checked",\n      "grid_s": [\n        2,\n        8,\n        36,\n        38,\n        40,\n        42\n      ],\n      "normalized_rgb_difference": [\n        0.1163386715282821,\n        0.1550361090938433,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 13263369,\n          "sha256": "1b66c99709bd687ec4501e335e2e60487fb468b362210bfac182b3dabfde8a5e",\n          "source_frames": [\n            48,\n            192,\n            864,\n            912,\n            960,\n            1008\n          ],\n          "frame_start_s": [\n            2.0,\n            8.0,\n            36.0,\n            38.0,\n            40.0,\n            42.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "0a700de66602b5150af0ddc62fad3ccbac35555c5ba78d4e597364bae6a4f2d3",\n            "b0dc6285a371cc5913e73a5d6dcf2588186aa402d5191a82018ebd68593055fb",\n            "6bc68ab0477926bbfe7320ff215b1bc9a3b023d6f06708ed400fdffbf43d1ee9",\n            "167477b4d95644c85a6e1fd3428b040075a9a978b57c730eb9f6bbf2ce391d31",\n            "583bc821b0544c459a8e5339aa3c091416bd39d315c454ccf35df6fb27cf17ea",\n            "9dc9e981a0bd53d25d30596f62f4a17bc0f84eedfb76a55c50d3815594a1f830"\n          ]\n        },\n        "b": {\n          "bytes": 13179586,\n          "sha256": "d18fad754703d8608aaad645da26d4a34ca2a67132bea66ef4b5533b2d37b1f5",\n          "source_frames": [\n            48,\n            192,\n            864,\n            912,\n            960,\n            1008\n          ],\n          "frame_start_s": [\n            2.0,\n            8.0,\n            36.0,\n            38.0,\n            40.0,\n            42.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "3ac43dd88cc114adc01eac322ea0dcc775babc4e68edf33bbc35b794683aee2f",\n            "fc3a08728c48d9343dcbaf2037bf281a25489413e39e903c7f46b3c78a42333d",\n            "6bc68ab0477926bbfe7320ff215b1bc9a3b023d6f06708ed400fdffbf43d1ee9",\n            "167477b4d95644c85a6e1fd3428b040075a9a978b57c730eb9f6bbf2ce391d31",\n            "583bc821b0544c459a8e5339aa3c091416bd39d315c454ccf35df6fb27cf17ea",\n            "9dc9e981a0bd53d25d30596f62f4a17bc0f84eedfb76a55c50d3815594a1f830"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0259",\n      "task": "HSR",\n      "status": "checked",\n      "grid_s": [\n        22,\n        24,\n        30,\n        32,\n        34,\n        36\n      ],\n      "normalized_rgb_difference": [\n        0.12353893074719599,\n        0.09419638354312918,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 20144782,\n          "sha256": "17b8f37fe3484646cfcfce19be92cbc32ad7c3a9a14fbde09be9e584587c1122",\n          "source_frames": [\n            528,\n            576,\n            720,\n            768,\n            816,\n            864\n          ],\n          "frame_start_s": [\n            22.0,\n            24.0,\n            30.0,\n            32.0,\n            34.0,\n            36.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "9fbe6db2b29c905f3d915c0c3a34f3aabe3fa3a428b146cb490ab6b92b858743",\n            "66c11b1023ef953b1605a2212b3bd089f3e33d11bce6673dcedee6a3e8ebf7fd",\n            "a5b67238bbaa1eae08487c99799243c4099574b96f689706946beb330acfa606",\n            "465c87264ae1413d3bb0c234515dc2b764b47afe4af77aa6e2d00cac14fc16de",\n            "bf84131ed26e32d816eb58ee02a30198f8c7bff59d0012aa5645c64e8fb2b68f",\n            "a4912c8b14054332c03aa47e05577ebc1b7bb4996d7df86655b1f78c17f096ad"\n          ]\n        },\n        "b": {\n          "bytes": 20325120,\n          "sha256": "b53459376e6138f03c8d3f57e8c55db0e8fcb6c8dcb05af49ae2bc3584573349",\n          "source_frames": [\n            528,\n            576,\n            720,\n            768,\n            816,\n            864\n          ],\n          "frame_start_s": [\n            22.0,\n            24.0,\n            30.0,\n            32.0,\n            34.0,\n            36.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "974e4841d7e4f3300c29ed176d22e137e81b9df5dd341677a47d138faa6b858f",\n            "c6828ef14fd69cad69e2a9c3525bac9c8e8af6574607395e3eb446f03bccec52",\n            "a5b67238bbaa1eae08487c99799243c4099574b96f689706946beb330acfa606",\n            "465c87264ae1413d3bb0c234515dc2b764b47afe4af77aa6e2d00cac14fc16de",\n            "bf84131ed26e32d816eb58ee02a30198f8c7bff59d0012aa5645c64e8fb2b68f",\n            "a4912c8b14054332c03aa47e05577ebc1b7bb4996d7df86655b1f78c17f096ad"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0196",\n      "task": "AIR",\n      "status": "insufficient_disjoint_earlier_slots",\n      "earlier_grid_s": [\n        0\n      ],\n      "recent_grid_s": [\n        2,\n        4,\n        6,\n        8\n      ]\n    },\n    {\n      "pair_id": "twinpast_0185",\n      "task": "AIR",\n      "status": "checked",\n      "grid_s": [\n        0,\n        2,\n        10,\n        12,\n        14,\n        16\n      ],\n      "normalized_rgb_difference": [\n        0.017549095255386105,\n        0.018344277011216006,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 4102201,\n          "sha256": "cad2075904aaad5577e1a9b60887708502e4a09f2829039420445eabd664dd2f",\n          "source_frames": [\n            0,\n            48,\n            240,\n            288,\n            336,\n            384\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            10.0,\n            12.0,\n            14.0,\n            16.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "3eaee29d075cc5ded5af7a2f4257d391d5334c184d16d040435a9282fb124c7d",\n            "8060ad1622d691ee24cb93511011aed4d0a3411d2828fe67f029bfd32b8caf35",\n            "43eed2d75749730936e015e9b0da77a32ff6693b62fe9d246f05b501149584fd",\n            "2dfaebf576dae6e423a49437d42543129a31405a88735f5a67873fb7829be38a",\n            "50f5455db1814ea7db6fc13dfe955f59dd32f26a0adc15dfabefd92b66641451",\n            "3b4e6fb633e5a6bcfbefb000686638e4e75f66af1a38c1f488cefb1833e35af0"\n          ]\n        },\n        "b": {\n          "bytes": 4122677,\n          "sha256": "f9b1001a8a8110ac6c49e7fda3e4c63f7ab1acdc5e07d0a3857f1bf7b526f8a2",\n          "source_frames": [\n            0,\n            48,\n            240,\n            288,\n            336,\n            384\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            10.0,\n            12.0,\n            14.0,\n            16.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "bac14025ba11782792a426cd2dd149eee70d693641fa3e985bcf0d891d4545dc",\n            "be522aec23fadd17e2607ca5eee03ad52a53ad400be88eeff7cbea1e22db92aa",\n            "43eed2d75749730936e015e9b0da77a32ff6693b62fe9d246f05b501149584fd",\n            "2dfaebf576dae6e423a49437d42543129a31405a88735f5a67873fb7829be38a",\n            "50f5455db1814ea7db6fc13dfe955f59dd32f26a0adc15dfabefd92b66641451",\n            "3b4e6fb633e5a6bcfbefb000686638e4e75f66af1a38c1f488cefb1833e35af0"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0214",\n      "task": "AIR",\n      "status": "checked",\n      "grid_s": [\n        0,\n        2,\n        38,\n        40,\n        42,\n        44\n      ],\n      "normalized_rgb_difference": [\n        0.03903457344872105,\n        0.039463810215444206,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 25963688,\n          "sha256": "1f4ec4b343fd0ae0ab92cf07d61df32c71e8f56485d5d3ce0ed8095cb42985e1",\n          "source_frames": [\n            0,\n            48,\n            912,\n            960,\n            1008,\n            1056\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            38.0,\n            40.0,\n            42.0,\n            44.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "34d7ee56544be90bd5a66c95ce6180e6df41e585cd4e9e595eb0b711207129bd",\n            "1a5cc77d2dcff19637ff1dcea4ac9f6f0762a3d24f9927530d5b152f9f4eda0f",\n            "c551aa44877768a39ba8a3ca35a7356436fd29b3e0804619a3287093a7201b1d",\n            "af4317a3219e2a70b61a4cfce46fb6a74e21040d0c62b8470f4f2b68b7289e57",\n            "f800c78548f52bd969fe60c6ce1095f41ee5f668d05b020ccbb6d029b0c17101",\n            "723f7ca19bdfb2baf60ba5b09f5d92821dffa39adb6934a56ebe8c25a6906e53"\n          ]\n        },\n        "b": {\n          "bytes": 25802733,\n          "sha256": "0682f3652c834a8868df6de722caae39ddac9fac6da6b9fcec08ed96b96b7c27",\n          "source_frames": [\n            0,\n            48,\n            912,\n            960,\n            1008,\n            1056\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            38.0,\n            40.0,\n            42.0,\n            44.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "7c1ba7613e7d48cb7917ffb1cdf6d56f6593555ec24c6f9877607b4d0b21f0db",\n            "2c3acd91ccf37c91a136f26b47f69f0a9ff206901be94f0eff17a6797a158e78",\n            "c551aa44877768a39ba8a3ca35a7356436fd29b3e0804619a3287093a7201b1d",\n            "af4317a3219e2a70b61a4cfce46fb6a74e21040d0c62b8470f4f2b68b7289e57",\n            "f800c78548f52bd969fe60c6ce1095f41ee5f668d05b020ccbb6d029b0c17101",\n            "723f7ca19bdfb2baf60ba5b09f5d92821dffa39adb6934a56ebe8c25a6906e53"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0079",\n      "task": "AIR",\n      "status": "checked",\n      "grid_s": [\n        6,\n        8,\n        78,\n        80,\n        82,\n        84\n      ],\n      "normalized_rgb_difference": [\n        0.018347302912934724,\n        0.020493864984265312,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 50061847,\n          "sha256": "9593bf6dd0ed8f11779767f627e5add09a935ef1a1854c4acda8649c389aad27",\n          "source_frames": [\n            144,\n            192,\n            1872,\n            1920,\n            1968,\n            2016\n          ],\n          "frame_start_s": [\n            6.0,\n            8.0,\n            78.0,\n            80.0,\n            82.0,\n            84.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "98cac78f9c88c598608e9d2824a56ca9c03f9f33a3831af332f1175e23af26dd",\n            "e4189ac7821ce3fa99f4e52de13f2e9c090cc6949e4d302f4dc6dc8e9c9e4d59",\n            "8b9550ba136e4c0dd73e309f7da71ade707bb7307e4c75a634fe70be97125ecb",\n            "1b1cfc5384a3c9abaed2e1a75d332314e0ef5f05df0d6f59ebfd814886e7b644",\n            "164a38204107b892ec9d9b902f7e8081e2c9a58e6b25e167b86e027babc253a6",\n            "03f668abf4d9ec7511d060843c35628ba7185c4d142059f5aa760c0695b16155"\n          ]\n        },\n        "b": {\n          "bytes": 49883514,\n          "sha256": "7141fb5b3895869f0b7d258aea43dee43c55989c162b7e23aa6b7c2cd005acb9",\n          "source_frames": [\n            144,\n            192,\n            1872,\n            1920,\n            1968,\n            2016\n          ],\n          "frame_start_s": [\n            6.0,\n            8.0,\n            78.0,\n            80.0,\n            82.0,\n            84.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "a0b22ed4c4ab8716203eae1ee60db2c113048ab9520831c2c81bdb86faa66387",\n            "2a8385e9bf42c7f13befa638ce60da9c51bf0387d662aafb5fb6daa672d239cf",\n            "8b9550ba136e4c0dd73e309f7da71ade707bb7307e4c75a634fe70be97125ecb",\n            "1b1cfc5384a3c9abaed2e1a75d332314e0ef5f05df0d6f59ebfd814886e7b644",\n            "164a38204107b892ec9d9b902f7e8081e2c9a58e6b25e167b86e027babc253a6",\n            "03f668abf4d9ec7511d060843c35628ba7185c4d142059f5aa760c0695b16155"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0236",\n      "task": "AIR",\n      "status": "checked",\n      "grid_s": [\n        0,\n        2,\n        32,\n        34,\n        36,\n        38\n      ],\n      "normalized_rgb_difference": [\n        0.06756743978455579,\n        0.0672045837367869,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 16063112,\n          "sha256": "930820bc91fc7d02d8c3abb78c39897be89d5ea42c5cd4d1010a8a874b5d323d",\n          "source_frames": [\n            0,\n            48,\n            768,\n            816,\n            864,\n            912\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            32.0,\n            34.0,\n            36.0,\n            38.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "3d83b1ed21e6618d2d4b7df1e190ee1a5b9661cb13f82fc4059538c6fbd67a47",\n            "5f2efbcdfc72b51b7d5161dd0696d834c909705034eeec183ac80380b51c0fc7",\n            "bada6290738792500a15bc7238f9841afcda840d25789f962d14467a038a722d",\n            "0c472ecb7f599ea6187b98d66d47211a9a16366331e555b9b9968b96ec371267",\n            "33962a108546ee21b480e0a79788a48866cc9c11c06753ecbaf3f0d345aa54a5",\n            "897eac61866ada1b466a10e54b1e00b8859bd393216b82a5338f7e0633530baa"\n          ]\n        },\n        "b": {\n          "bytes": 15999399,\n          "sha256": "b9c9886edbced749b9bc6d6e326f564ab10bce9f5c34238c5441b99060f6f84a",\n          "source_frames": [\n            0,\n            48,\n            768,\n            816,\n            864,\n            912\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            32.0,\n            34.0,\n            36.0,\n            38.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "fa4858f7e4fc2e4f31eb09af1f2a8e0357c09c835bab1db5088cc2ce5f51b376",\n            "934786e60c08527e9608279240a907c3eb824ceff81a3d6c0d60d000c3a232cc",\n            "bada6290738792500a15bc7238f9841afcda840d25789f962d14467a038a722d",\n            "0c472ecb7f599ea6187b98d66d47211a9a16366331e555b9b9968b96ec371267",\n            "33962a108546ee21b480e0a79788a48866cc9c11c06753ecbaf3f0d345aa54a5",\n            "897eac61866ada1b466a10e54b1e00b8859bd393216b82a5338f7e0633530baa"\n          ]\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0036",\n      "task": "AIR",\n      "status": "checked",\n      "grid_s": [\n        0,\n        2,\n        36,\n        38,\n        40,\n        42\n      ],\n      "normalized_rgb_difference": [\n        0.024057053376906317,\n        0.025663618070685065,\n        0.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      "decoded": {\n        "a": {\n          "bytes": 17576106,\n          "sha256": "4fa6c986d8f9ccd3ca4446f1443e27a272f503cfaec5f48ffa83c2affc11efd9",\n          "source_frames": [\n            0,\n            48,\n            864,\n            912,\n            960,\n            1008\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            36.0,\n            38.0,\n            40.0,\n            42.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "a7087d59a82eaa534cdeb7a7e4658b90c567fbb821006287ddb06f9525d645bf",\n            "9ff15e013914ec9e72fa260a1a429e9e521ea84a201e9deda889e6582d616f9d",\n            "10bec95e44330134059179acca695988028a13c452fea71cc037b17ed6f70956",\n            "8643e484463441ab8881b171b2517b2c66f707c5a483c8b2ab89fe70717cf8cb",\n            "90fd9562473536c1145fbb0db0e426d3da7d595789ddb0e66e42743edcd87ee2",\n            "476b94ea69d0908dd0e42004acd3d621a48ad5e2127846ede605c025fad6e3d3"\n          ]\n        },\n        "b": {\n          "bytes": 17480443,\n          "sha256": "8ed2e5a5d7dec040cf207eadfa0523ab92498aee34e2762b468779f40fdbee92",\n          "source_frames": [\n            0,\n            48,\n            864,\n            912,\n            960,\n            1008\n          ],\n          "frame_start_s": [\n            0.0,\n            2.0,\n            36.0,\n            38.0,\n            40.0,\n            42.0\n          ],\n          "early_samples_inside": [\n            true,\n            true\n          ],\n          "decoded_rgb_sha256": [\n            "7d110980d012aa21e2329db52881007a5664d83e8a30c8a486b45bf80c1b12a4",\n            "b8a757c00933198580b36a2714ba29f91470e5255cb744ea70275d0e1a4795d2",\n            "10bec95e44330134059179acca695988028a13c452fea71cc037b17ed6f70956",\n            "8643e484463441ab8881b171b2517b2c66f707c5a483c8b2ab89fe70717cf8cb",\n            "90fd9562473536c1145fbb0db0e426d3da7d595789ddb0e66e42743edcd87ee2",\n            "476b94ea69d0908dd0e42004acd3d621a48ad5e2127846ede605c025fad6e3d3"\n          ]\n        }\n      }\n    }\n  ]\n}\n'}
EXPECTED = {'pinned_source/inference/llavaov_hermes.py': '45fee483fbc37fe65433c84146eba42fc18dae40f6878c7a2f50fd8e55a387f0', 'pinned_source/inference/abstract_hermes.py': 'cbf62ad9f6dada32db598d7c37c27f995bbb37375e12a282a88cf4426f495dc0', 'pinned_source/inference/reindex_1d.py': '22ca0dee63cd7a4d00161f1fc0485143a57145862b081555598b904bbc95e0b0', 'provenance/twinpast_runner.py': '5f4e15c54168502763fb8ad2054e6c701e7440a09295b01240ef05e3865066f0', 'provenance/bundle.json': '451af6336af2e67906a9d9f3263cc22d210adaaf6ecfa8c3a8ad826a68d3d004'}

for relative, contents in SOURCES.items():
    raw = contents.encode('utf-8')
    assert hashlib.sha256(raw).hexdigest() == EXPECTED[relative]
    if relative.endswith('.py'):
        ast.parse(contents, filename=relative)
    if not VALIDATE_ONLY:
        target = WORK / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.read_bytes() != raw:
            raise ValueError(f'Existing file differs at {target}. Preserve it and choose a fresh WORK directory.')
        if not target.is_file():
            target.write_bytes(raw)
print('EMBEDDED_SOURCE_AND_MANIFEST_PASS', BASE_REVISION)


EMBEDDED_SOURCE_AND_MANIFEST_PASS 2a1385e0abc94debb64761b0465713c799a81c1c


In [5]:
ENV_PROBE = r"""
import sys, torch, transformers, tokenizers, huggingface_hub, pandas, decord, logzero
import json, importlib.metadata as md
assert sys.version_info[:2] == (3, 12)
assert torch.__version__ == '2.10.0+cu128'
assert transformers.__version__ == '4.45.0.dev0'
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert '66bc4def9505fa7c7fe4aa7a248c34a026bb552b' in str(origin), 'Transformers source revision differs'
assert tokenizers.__version__ == '0.19.1' and huggingface_hub.__version__ == '0.34.4'
assert torch.cuda.is_available(), 'PyTorch cannot use CUDA; check the container driver'
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
import inspect
assert 'seq_len' in inspect.signature(Qwen2RotaryEmbedding.forward).parameters
x = torch.ones((16, 16), device='cuda')
assert float((x @ x)[0, 0]) == 16
print('PINNED_ENV_PASS', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
"""
if not VALIDATE_ONLY:
    candidate_python = REUSE/'venv/bin/python'
    if candidate_python.is_file() and probe([candidate_python, '-c', ENV_PROBE], ENV).returncode == 0:
        PYTHON = candidate_python
        print('REUSING_VERIFIED_PINNED_ENVIRONMENT', PYTHON, flush=True)
    PIP_ENV = package_environment(ENV)
    if not PYTHON.is_file() or probe([PYTHON, '-c', ENV_PROBE], ENV).returncode:
        if probe([sys.executable, '-m', 'uv', '--version'], ENV).returncode:
            attempt = probe([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                             'https://pypi.tuna.tsinghua.edu.cn/simple'], PIP_ENV)
            if attempt.returncode:
                logged([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                        'https://mirrors.aliyun.com/pypi/simple'], 'setup.log', PIP_ENV)
        UV = [sys.executable, '-m', 'uv']
        if not PYTHON.is_file():
            logged(UV + ['venv', VENV, '--python', '3.12', '--seed'], 'setup.log', ENV)
        def install(packages, extra=()):
            install_packages(UV, PYTHON, packages, ENV, extra)
        install(['torch @ https://download-r2.pytorch.org/whl/cu128/torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl'])
        install(['numpy==1.26.4', 'tokenizers==0.19.1', 'huggingface-hub==0.34.4',
                 'accelerate==0.34.2', 'decord==0.6.0', 'logzero', 'pillow',
                 'sentencepiece', 'safetensors', 'packaging', 'filelock', 'regex',
                 'requests', 'pyyaml', 'tqdm', 'fsspec', 'protobuf', 'einops', 'psutil', 'pandas'])
        # Download GitHub source with turbo, then install the local archive using
        # direct package networking for any isolated build dependencies.
        tf_archive = WORK/'downloads'/f'transformers-{TF_SHA}.zip'
        tf_archive.parent.mkdir(parents=True, exist_ok=True)
        if not tf_archive.is_file() or not zipfile.is_zipfile(tf_archive):
            fetch_tf = r"""
import sys, urllib.request, zipfile, os, time
from pathlib import Path
dest = Path(sys.argv[2]); temp = dest.with_suffix('.download')
for attempt in range(4):
    try:
        with urllib.request.urlopen(sys.argv[1], timeout=180) as source, temp.open('wb') as target:
            while True:
                block = source.read(1024 * 1024)
                if not block: break
                target.write(block)
        assert zipfile.is_zipfile(temp), 'Transformers source is not a ZIP'
        os.replace(temp, dest)
        break
    except Exception:
        if attempt == 3: raise
        time.sleep(3 * (attempt + 1))
"""
            logged([sys.executable, '-c', fetch_tf,
                    f'https://codeload.github.com/huggingface/transformers/zip/{TF_SHA}', tf_archive],
                   'source_download.log', ENV)
        install([str(tf_archive)], ['--no-deps'])
    logged([PYTHON, '-c', ENV_PROBE], 'environment.log', ENV)
    ENV['PYTHONPATH'] = str(REPO) + os.pathsep + str(REPO/'scripts')
    logged([PYTHON, '-m', 'pip', 'freeze'], 'environment_freeze.txt', ENV)


REUSING_VERIFIED_PINNED_ENVIRONMENT /root/autodl-tmp/hermes_causal_pilot/venv/bin/python
SETUP_STAGE environment.log
PINNED_ENV_PASS 2.10.0+cu128 4.45.0.dev0 NVIDIA GeForce RTX 4090
SETUP_STAGE environment_freeze.txt
accelerate==0.34.2
certifi==2026.10.23
charset-normalizer==3.5.2
cuda-bindings==12.9.4
cuda-pathfinder==1.9.1
decord==0.6.0
einops==0.8.2
filelock==4.2.12
fsspec==2026.9.0
hf-xet==1.8.0
huggingface-hub==0.34.4
idna==3.20
Jinja2==3.1.6
logzero==1.7.0
MarkupSafe==3.0.4
mpmath==1.3.0
networkx==3.7
numpy==1.26.4
nvidia-cublas-cu12==12.8.4.1
nvidia-cuda-cupti-cu12==12.8.90
nvidia-cuda-nvrtc-cu12==12.8.93
nvidia-cuda-runtime-cu12==12.8.90
nvidia-cudnn-cu12==9.10.2.21
nvidia-cufft-cu12==11.3.3.83
nvidia-cufile-cu12==1.13.1.3
nvidia-curand-cu12==10.3.9.90
nvidia-cusolver-cu12==11.7.3.90
nvidia-cusparse-cu12==12.5.8.93
nvidia-cusparselt-cu12==0.7.1
nvidia-nccl-cu12==2.27.5
nvidia-nvjitlink-cu12==12.8.93
nvidia-nvshmem-cu12==3.4.5
nvidia-nvtx-cu12==12.8.90
packaging==26.3
pandas==3.

In [6]:
MODEL_DOWNLOAD = r"""
import json, sys
from pathlib import Path
from huggingface_hub import snapshot_download
from safetensors import safe_open
path = Path(snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], cache_dir=sys.argv[3],
    allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.vocab', 'tokenizer.*', 'merges.txt'],
    ignore_patterns=['onnx/*', '*.onnx', '*.bin', '*.msgpack'], max_workers=4))
assert path.name == sys.argv[2] and (path/'config.json').is_file()
files = list(path.glob('*.safetensors'))
assert files
for file in files:
    with safe_open(str(file), framework='pt', device='cpu') as handle:
        assert list(handle.keys())
Path(sys.argv[4]).write_text(json.dumps(str(path)), encoding='utf-8')
print('MODEL_REVISION_PASS', path.name)
"""
if not VALIDATE_ONLY:
    pointer = WORK/'model_path.json'
    local_model = Path(ENV['HF_HOME'])/'hub'/'models--llava-hf--llava-onevision-qwen2-0.5b-ov-hf'/'snapshots'/MODEL_SHA
    local_env = {**ENV, 'HF_HUB_OFFLINE': '1'}
    use_cache = local_model.is_dir() and probe(
        [PYTHON, '-c', MODEL_DOWNLOAD, MODEL_ID, MODEL_SHA, Path(ENV['HF_HOME'])/'hub', pointer], local_env
    ).returncode == 0
    if use_cache:
        print('REUSING_PINNED_MODEL', local_model)
    failures = []
    for host in ('https://hf-mirror.com', 'https://huggingface.co'):
        if use_cache:
            break
        try:
            logged([PYTHON, '-c', MODEL_DOWNLOAD, MODEL_ID, MODEL_SHA,
                    Path(ENV['HF_HOME'])/'hub', pointer], 'model_download.log', {**ENV, 'HF_ENDPOINT': host})
            break
        except RuntimeError as exc:
            failures.append(type(exc).__name__)
            print('Model endpoint failed; trying fallback', flush=True)
    else:
        raise RuntimeError('Model download failed on both endpoints; rerun to resume cached files')
    MODEL_PATH = Path(json.loads(pointer.read_text(encoding='utf-8')))
    assert MODEL_PATH.name == MODEL_SHA


REUSING_PINNED_MODEL /root/autodl-tmp/hermes_causal_pilot/cache/huggingface/hub/models--llava-hf--llava-onevision-qwen2-0.5b-ov-hf/snapshots/74dd0bf867a4cda7950c17663794267c60cf4b40


## 2. Acquire clips, run the preflight and score the nine pairs


In [7]:
ARTIFACT = 'HERMES_twinpast_diagnostic_evidence.zip'

if not VALIDATE_ONLY:
    (WORK/'launch.json').write_text(json.dumps(dict(started_utc=datetime.now(timezone.utc).isoformat(),
        source_revision=BASE_REVISION, model_revision=MODEL_SHA, transformers_revision=TF_SHA,
        embedded_sha256=EXPECTED, gpu=smi.stdout.strip()), indent=2), encoding='utf-8')
    env = {**ENV, 'PYTHONDONTWRITEBYTECODE': '1', 'HF_HUB_OFFLINE': '1'}
    logged([PYTHON, '-u', RUNNER, '--work', WORK, '--model', MODEL_PATH, '--media-reuse', MEDIA_REUSE],
           'twinpast.log', env, REPO)
else:
    print('VALIDATION_PASS: setup and embedded sources checked without GPU, network or file writes')


SETUP_STAGE twinpast.log
MEDIA_READY data.jsonl 0.2 MiB
MEDIA_PLAN 18 pinned clips, up to 361.6 MiB not present
MEDIA_READY videos/twinpast_0208/A.mp4 18.5 MiB
MEDIA_PROGRESS videos/twinpast_0094/A.mp4 14.2 MiB
MEDIA_READY videos/twinpast_0208/B.mp4 18.5 MiB
MEDIA_READY videos/twinpast_0095/A.mp4 12.6 MiB
MEDIA_READY videos/twinpast_0094/A.mp4 22.4 MiB
MEDIA_READY videos/twinpast_0095/B.mp4 12.6 MiB
MEDIA_READY videos/twinpast_0259/A.mp4 19.2 MiB
MEDIA_PROGRESS videos/twinpast_0094/B.mp4 3.8 MiB
MEDIA_PROGRESS videos/twinpast_0094/B.mp4 7.8 MiB
MEDIA_PROGRESS videos/twinpast_0259/B.mp4 11.0 MiB
MEDIA_PROGRESS videos/twinpast_0185/A.mp4 3.0 MiB
MEDIA_READY videos/twinpast_0185/A.mp4 3.9 MiB
MEDIA_READY videos/twinpast_0094/B.mp4 21.9 MiB
MEDIA_READY videos/twinpast_0259/B.mp4 19.4 MiB
MEDIA_READY videos/twinpast_0185/B.mp4 3.9 MiB
MEDIA_READY videos/twinpast_0214/B.mp4 24.6 MiB
MEDIA_RETRY videos/twinpast_0214/A.mp4 host=huggingface.co direct=False reason=ReadTimeout
MEDIA_RETRY videos/

## 3. Download the evidence ZIP

After a runner error, run this cell manually to retrieve failure evidence.


In [8]:

if not VALIDATE_ONLY:
    from IPython.display import FileLink, display
    artifact = WORK/ARTIFACT
    if artifact.is_file():
        link = Path.cwd()/ARTIFACT
        if artifact.resolve() != link.resolve():
            shutil.copyfile(artifact, link)
        print((WORK/'completion.json').read_text(encoding='utf-8'))
        display(FileLink(ARTIFACT))
    else:
        print('No runner ZIP yet. Inspect setup logs under', WORK/'logs')


{
  "status": "complete",
  "pairs": 9,
  "scores": 72
}



/root/autodl-tmp/HERMES-reproduction/HERMES_twinpast_diagnostic_evidence.zip